In [1]:
# %pip install -q sagemaker boto3 pandas pyarrow matplotlib scikit-learn scipy   # once per kernel; pin "sagemaker<3" to force the v2 API

import os, io, re, gc, json, time, tarfile, ctypes
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import boto3, sagemaker
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ Config ----
BUCKET          = 'visilant-data-transfer-s3bucket-6ocumykwtnur'
RAW_CSV         = 'keratitis_images_azure.csv'       # original export: microbiology (visit-level), illumination, device, site, dates
EYE_CSV         = 'keratitis_labels 2.csv'           # per-eye export: corneal pathology L/R, morphology, symptoms days, risk factor, VA
BASE_PREFIX     = 'diffuse-512'                      # earlier staged sets: used only to inherit patient splits
V3_PREFIX       = 'active-bfv-diffuse'
V4_PREFIX       = 'v4-diffuse-1024'                  # the 1024-px images (diffuse from v4; blue-light pairs are added here by this notebook)
DATA_PREFIX     = V4_PREFIX
LABELS_FILE     = 'full_train_labels_meta.csv'       # v4 manifest name (split inheritance)
V7_LABELS       = 'labels_v7.csv'                    # this run's manifest (clean eye-visits only)
V7_ALL          = 'labels_v7_all.csv'                # every row incl. fellow / ambiguous (analysis only)
ARCHIVE_PREFIX  = 'v7-archive-384'                   # whole archive at 384 px for self-supervised pretraining
ARCHIVE_MANIFEST = 'archive_manifest.csv'
MAX_ARCHIVE     = 0                                  # 0 = every image in the bucket root; set e.g. 30000 to cap staging time
FORCE_REBUILD   = False
EXPERIMENT_ROOT = 'experiments-v7'
LIMITS          = {'ml.g5.4xlarge': 2, 'ml.g5.2xlarge': 4, 'ml.g4dn.2xlarge': 2}     # approved quotas (ap-south-1)
FRAMEWORK_VERSION, PY_VERSION = '2.1', 'py310'
RUNS_DIR        = os.path.expanduser('~/runs_v7')    # LOCAL disk
V6_RUNS_DIR     = os.path.expanduser('~/runs_v6')    # if present, v6 runs are rescored on the v7 rows at visit level
LEDGER          = f'{RUNS_DIR}/ledger.json'
TARGETS         = {'bf': 0.88, 'bfv': 0.90}
SPLIT_SEED, TARGET_SHORT_SIDE, ARCHIVE_SHORT_SIDE, JPEG_QUALITY = 42, 1024, 384, 90
N_FOLDS         = 5
SEASONS = {12: 'winter', 1: 'winter', 2: 'winter', 3: 'summer', 4: 'summer', 5: 'summer',
           6: 'monsoon', 7: 'monsoon', 8: 'monsoon', 9: 'post_monsoon', 10: 'post_monsoon', 11: 'post_monsoon'}
CLEAN_STATUS    = ['infected', 'infected_propagated', 'single_unmatched']

for d in ('src', RUNS_DIR):
    os.makedirs(d, exist_ok=True)
try:
    ctypes.CDLL('libc.so.6').mallopt(-8, 2)          # cap malloc arenas: image-decoding threads otherwise balloon the kernel's RSS
except Exception:
    pass
pid_str = lambda s: s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
norm = lambda s: s.astype(str).str.strip()
strip_key = lambda s: s.astype(str).str.strip().str.lower().str.replace(r'\.(jpg|jpeg|png)$', '', regex=True)
s3 = boto3.client('s3')
sm = boto3.client('sagemaker')
REGION = boto3.Session().region_name

try:
    from sagemaker import get_execution_role, Session          # SDK v2
    SDK_V3 = False
except ImportError:
    from sagemaker.core.helper.session_helper import get_execution_role, Session   # SDK v3
    SDK_V3 = True
ROLE = get_execution_role()
SM_SESSION = Session()
SM_SESSION.default_bucket_prefix = (SM_SESSION.default_bucket_prefix or '').strip('/') or None
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', 44)
print(f'SageMaker SDK {"v3" if SDK_V3 else "v2"} | region {REGION} | role {ROLE.split("/")[-1]} | runs -> {RUNS_DIR}')
print(f'started {datetime.now():%Y-%m-%d %H:%M}')

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
SageMaker SDK v3 | region ap-south-1 | role AmazonSageMakerAdminIAMExecutionRole | runs -> /home/sagemaker-user/runs_v7
started 2026-10-15 18:14


## 1. Labels

**1a. Organism labels** (unchanged since v4): Bacterial / Fungal from microbiology only; Viral from `dx_category == 'Viral'` with no growth. Both diffuse and blue-light rows are kept here (blue rows become the pairs in 1c).

In [2]:
raw = pd.read_csv(RAW_CSV, low_memory=False)
raw['id'] = norm(raw['id']); raw['patient_id'] = pid_str(raw['patient_id'])
mr, mc, dx = norm(raw['micro_recovered']), norm(raw['micro_category']), norm(raw['dx_category'])
CONTRADICTORY_DX = ['Mixed', 'Viral', 'Acanthamoeba', 'Microsporidia', 'Other']
bf = np.where(mr.isin(['Bacterial', 'Fungal']), mr, np.where(mc.isin(['Bacterial', 'Fungal']), mc, None))
bf = np.where(pd.Series(bf).notna() & ~dx.isin(CONTRADICTORY_DX), bf, None)
raw['label'] = np.where(pd.Series(bf).notna(), bf, np.where(dx.eq('Viral') & ~mr.isin(['Bacterial', 'Fungal']) & ~mc.isin(['Bacterial', 'Fungal', 'Mixed']), 'Viral', None))
raw['determined'] = (mc.isin(['Bacterial', 'Fungal']) | raw['label'].eq('Viral')).astype(int)
raw['illum'] = norm(raw['illumination']).str.lower()
raw['device_n'] = norm(raw['device']).str.lower()
raw['site_n'] = norm(raw['site']).str.upper()
raw['eye_l'] = norm(raw['eye']).str.lower()
raw['month'] = pd.to_numeric(raw['created_at'].astype(str).str.slice(5, 7), errors='coerce')
raw['img_date'] = pd.to_datetime(raw['created_at'], errors='coerce', utc=True)
raw['key'] = strip_key(raw['id'])
raw['is_blue'] = raw['illum'].str.contains('blue', na=False)
NON_EYE = r'paper|form|consent|microbiolog|result|report|prescription'
raw['non_eye'] = raw['type'].astype(str).str.lower().str.contains(NON_EYE, na=False) if 'type' in raw.columns else False

lab = raw[raw['label'].notna() & raw['illum'].eq('diffuse') & ~raw['non_eye']].copy()
print(f'labelled diffuse rows: {len(lab):,} from {lab.patient_id.nunique():,} patients; labelled blue-light rows: {int((raw.label.notna() & raw.is_blue).sum()):,}')
print(pd.crosstab([lab['site_n'], lab['device_n']], lab['label']).to_string(), '\n')
print('determined-visit share by label:', lab.groupby('label')['determined'].mean().round(3).to_dict())

labelled diffuse rows: 40,789 from 5,138 patients; labelled blue-light rows: 39,011
label              Bacterial  Fungal  Viral
site_n device_n                            
SCEH   slit_lamp        3323    4294   1147
       smartphone       2981    3625   1037
SNC    slit_lamp        3501    7690   1391
       smartphone       3159    7291   1350 

determined-visit share by label: {'Bacterial': 0.503, 'Fungal': 0.469, 'Viral': 1.0}


**1b. Eye-level status and clinical features** (as v6): `infected` / `fellow` / `single_unmatched` / `ambiguous`, propagation of the infected side across a patient's visits, and days of symptoms / risk factor / VA of the photographed eye.

In [3]:
ex = pd.read_csv(EYE_CSV, low_memory=False, dtype=str)
def find_col(df, *needles):
    for c in df.columns:
        k = ' '.join(c.lower().split())
        if all(n in k for n in needles): return c
    raise KeyError(needles)
COLS = dict(path_L=find_col(ex, 'corneal path', 'left'), path_R=find_col(ex, 'corneal path', 'right'),
            sym_L=find_col(ex, 'symptoms days', 'left'), sym_R=find_col(ex, 'symptoms days', 'right'),
            rf_L=find_col(ex, 'risk factor', 'left'), rf_R=find_col(ex, 'risk factor', 'right'),
            va_L=find_col(ex, 'valeft'), va_R=find_col(ex, 'varight'))
ex['key'] = strip_key(ex['image_name']); ex['eye_x'] = norm(ex['eye']).str.lower()
n_dup = ex.duplicated('key').sum(); ex = ex.drop_duplicates('key')
isL = ex['eye_x'] == 'left'
pick = lambda L, R: pd.Series(np.where(isL, ex[COLS[L]], ex[COLS[R]]), index=ex.index)
ex['img_path'], ex['oth_path'] = pick('path_L', 'path_R'), pick('path_R', 'path_L')
ex['sym_days'], ex['risk'], ex['va'] = pick('sym_L', 'sym_R'), pick('rf_L', 'rf_R'), pick('va_L', 'va_R')
ex['ex_date'] = pd.to_datetime(ex['image_created_at'], errors='coerce', utc=True)
def simp(s):
    s = s.astype(str).str.strip()
    return np.select([s.isin(['', 'nan', 'None']), s == 'Normal', s.str.contains('Active corneal infection', na=False)], ['(missing)', 'Normal', 'Active'], default='Other')
ex['img_path_s'], ex['oth_path_s'] = simp(ex['img_path']), simp(ex['oth_path'])
print(f'per-eye export: {len(ex):,} unique images ({n_dup} duplicate names dropped), {str(ex.ex_date.min())[:10]} -> {str(ex.ex_date.max())[:10]}')

def va_logmar(v):
    s = str(v).strip().lower()
    if s in ('', 'nan', 'n/a', 'na', 'none', '-'): return np.nan
    m = re.match(r'^(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)', s)
    if m:
        num, den = float(m.group(1)), float(m.group(2))
        return float(np.log10(den / num)) if num > 0 and den > 0 else np.nan
    if 'no light' in s or 'no perception' in s or s in ('nlp', 'npl', 'no pl'): return 3.0
    if 'light' in s or 'perception' in s or s in ('pl', 'lp', 'pl+', 'pl +'): return 2.6
    if 'hand' in s or s.startswith('hm'): return 2.3
    if 'count' in s or s.startswith('cf') or 'finger' in s: return 1.9
    return np.nan
RF_RULES = [('vegetative', r'vegetat|plant|leaf|leaves|stick|thorn|sugarcane|sugar cane|paddy|wood|branch|grass|crop|agricul|straw|husk|rice|wheat|tree'),
            ('insect', r'insect|fly\b|bee\b|mosquito|bug\b|wing'),
            ('trauma_other', r'trauma|injur|foreign body|\bfb\b|fall\b|dust|stone|nail|finger|scratch|blunt|metal|iron|cement|mud|sand|soil|cow|tail|accident'),
            ('contact_lens', r'contact lens|\bcl\b|lens wear'), ('dry_eye', r'\bdry'), ('steroid', r'steroid|dexa|predni|betameth'),
            ('diabetes', r'diabet|\bdm\b'), ('prior_surgery', r'surgery|surgical|keratoplasty|graft|cataract|phaco|lasik|\bpkp\b|\bdalk\b|post[- ]?op'),
            ('ocular_surface', r'blephar|trichiasis|entropion|lagoph|exposure|neurotroph|bullous|keratopathy|dacryo|\bnld|lacrimal|chronic|previous infection|recurr'),
            ('none', r'no risk|^none$|^nil$|^no$|not known|unknown')]
RF_CATS = [c for c, _ in RF_RULES] + ['other']
def risk_categories(v):
    s = str(v).strip().lower()
    if s in ('', 'nan', 'n/a', 'na', '-'): return None
    hits = [c for c, pat in RF_RULES if re.search(pat, s)]
    if 'trauma_other' in hits and ({'vegetative', 'insect'} & set(hits)): hits.remove('trauma_other')
    return hits or ['other']
ex['sym_days_n'] = pd.to_numeric(ex['sym_days'].astype(str).str.extract(r'(\d+(?:\.\d+)?)')[0], errors='coerce')
ex['va_logmar'] = ex['va'].map(va_logmar)
ex['risk_cats'] = ex['risk'].map(risk_categories)

m = lab.merge(ex[['key', 'eye_x', 'img_path_s', 'oth_path_s', 'ex_date', 'sym_days_n', 'va_logmar', 'risk_cats']], on='key', how='left')
m['present'] = m['img_path_s'].notna()
m['visit_id'] = m['visit_id'].astype(str).where(m['visit_id'].notna(), 'v_' + m['id'].astype(str))
agree = (m.loc[m.present, 'eye_x'] == m.loc[m.present, 'eye_l']).mean()
print(f'labelled rows found in the per-eye export: {m.present.mean():.1%}; eye column agrees on {agree:.1%} of them')
m['eyes_at_visit'] = m.groupby(['patient_id', 'visit_id'])['eye_l'].transform('nunique')
m['eye_inf'] = m['present'] & (m['img_path_s'] != 'Normal')
m['eye_any_inf'] = m.groupby(['patient_id', 'visit_id', 'eye_l'])['eye_inf'].transform('max').fillna(False).astype(bool)
m['visit_any_inf'] = m.groupby(['patient_id', 'visit_id'])['eye_inf'].transform('max').fillna(False).astype(bool)
m['eye_status'] = np.select(
    [m['eye_any_inf'], m['present'] & (m['img_path_s'] == 'Normal'), (m['eyes_at_visit'] >= 2) & m['visit_any_inf'], m['eyes_at_visit'] == 1],
    ['infected', 'fellow', 'fellow', 'single_unmatched'], default='ambiguous')
sides = m[m.eye_status == 'infected'].groupby('patient_id')['eye_l'].agg(lambda s: frozenset(s[s.isin(['left', 'right'])]))
m['known_sides'] = m['patient_id'].map(sides)
amb, ks = m['eye_status'] == 'ambiguous', m['known_sides']
one_side = amb & ks.map(lambda v: isinstance(v, frozenset) and len(v) == 1) & m['eye_l'].isin(['left', 'right'])
both_sides = amb & ks.map(lambda v: isinstance(v, frozenset) and len(v) == 2)
m.loc[one_side & (m['eye_l'] == ks.map(lambda v: next(iter(v)) if isinstance(v, frozenset) and len(v) == 1 else None)), 'eye_status'] = 'infected_propagated'
m.loc[one_side & (m['eye_status'] == 'ambiguous'), 'eye_status'] = 'fellow_propagated'
m.loc[both_sides, 'eye_status'] = 'infected_propagated'
print('\nimages by eye_status:'); print(m['eye_status'].value_counts().to_string())

ref = (m[m.present & m['eye_inf']].sort_values('ex_date').groupby('patient_id')
       .agg(ref_date=('ex_date', 'first'), ref_sym=('sym_days_n', 'first'), ref_risk=('risk_cats', 'first')))
m = m.merge(ref, on='patient_id', how='left')
carry = ~m.present & m['eye_status'].isin(['infected_propagated', 'single_unmatched', 'ambiguous'])
elapsed = (m['img_date'] - m['ref_date']).dt.days
m['clin_sym_days'] = np.where(m.present, m['sym_days_n'], np.where(carry & elapsed.notna(), m['ref_sym'] + elapsed.clip(lower=0), np.nan))
m['clin_risk'] = [r if p else (rr if c else None) for p, c, r, rr in zip(m.present, carry, m['risk_cats'], m['ref_risk'])]
m['clin_risk'] = m['clin_risk'].where(m['clin_risk'].map(lambda v: isinstance(v, list)), None)
m['clin_va'] = np.where(m.present, m['va_logmar'], np.nan)
m['clin_source'] = np.select([m.present, carry & m['ref_date'].notna()], ['own record', 'carried over'], default='none')
print('\nclinical coverage by eye_status:'); print(m.assign(sym=m.clin_sym_days.notna(), rf=m.clin_risk.notna(), va=m.clin_va.notna()).groupby('eye_status')[['sym', 'rf', 'va']].mean().round(3).to_string())

per-eye export: 30,372 unique images (9 duplicate names dropped), 2024-05-10 -> 2026-04-13
labelled rows found in the per-eye export: 20.8%; eye column agrees on 100.0% of them

images by eye_status:
eye_status
ambiguous              15207
infected                8719
fellow                  8520
infected_propagated     4195
fellow_propagated       4106
single_unmatched          42

clinical coverage by eye_status:
                       sym     rf     va
eye_status                              
ambiguous            0.000  0.000  0.000
fellow               0.000  0.000  0.005
fellow_propagated    0.000  0.000  0.000
infected             0.662  0.661  0.967
infected_propagated  0.888  0.887  0.000
single_unmatched     0.286  0.286  0.000


**1c. Morphology auxiliary targets, blue-light pairs, visit order.** Twenty binary morphology targets are parsed from the per-eye record of the photographed eye (`aux_*`; NaN = unknown, masked in the loss). They are *never* inputs. Each labelled diffuse image is paired with the blue-light image of the same (patient, visit, eye) — same device preferred, nearest in time. `is_index` marks each patient's earliest labelled visit.

In [4]:
MC = {k: find_col(ex, *n) for k, n in {
    'hypo_L': ('hypopyon', 'left'), 'hypo_R': ('hypopyon', 'right'), 'epi_L': ('epithelial defect', 'left'), 'epi_R': ('epithelial defect', 'right'),
    'diam_L': ('infiltrate', 'diameter', 'left'), 'diam_R': ('infiltrate', 'diameter', 'right'), 'depth_L': ('infiltrate', 'depth', 'left'), 'depth_R': ('infiltrate', 'depth', 'right'),
    'loc_L': ('infiltrate', 'center', 'left'), 'loc_R': ('infiltrate', 'center', 'right'), 'limb_L': ('infiltrate', 'limbus', 'left'), 'limb_R': ('infiltrate', 'limbus', 'right'),
    'spec_L': ('specific exam', 'left'), 'spec_R': ('specific exam', 'right')}.items()}
side = lambda k: pd.Series(np.where(isL, ex[MC[k + '_L']], ex[MC[k + '_R']]), index=ex.index).astype(str).str.strip().str.lower()
blank = lambda s: s.isin(['', 'nan', 'none'])
def yes_no(s, flag):
    """Specific-exam field: 'A - No, B - Yes, ...' lists or bare lists of present findings or 'No specific Exam findings'."""
    out = pd.Series(np.nan, index=s.index)
    listed = s.str.contains(r' - (?:yes|no)', regex=True)
    items = s[listed].str.findall(r'([^,]+?)\s*-\s*(yes|no)')
    out[listed] = items.map(lambda it: next((1.0 if v == 'yes' else 0.0 for k, v in it if flag in k), np.nan))
    bare = ~listed & ~blank(s)
    out[bare] = np.where(s[bare].str.contains('no specific'), 0.0, s[bare].str.contains(flag).astype(float))
    return out
def ordinal(s, rules):
    out = pd.Series(np.nan, index=s.index)
    for pat, v in rules:
        hit = s.str.contains(pat, regex=True) & out.isna(); out[hit] = v
    return out
hypo, epi, diam, depth, loc, limb, spec = side('hypo'), side('epi'), side('diam'), side('depth'), side('loc'), side('limb'), side('spec')
mm = pd.to_numeric(hypo.str.extract(r'(\d+(?:\.\d+)?)\s*mm')[0], errors='coerce')
A = pd.DataFrame(index=ex.index)
A['hypopyon'] = np.where(blank(hypo), np.nan, np.where(hypo.str.contains(r'- no'), 0.0, np.where(mm.notna() | hypo.str.contains('yes'), 1.0, np.nan)))
A['hypopyon_ge2mm'] = np.where(blank(hypo), np.nan, np.where(hypo.str.contains(r'- no'), 0.0, np.where(mm.notna(), (mm >= 2).astype(float), np.nan)))
epi_o = ordinal(epi, [(r'no epithelial', 0), (r'>\s*0 to', 1), (r'2 to', 2), (r'>\s*6', 3)])
A['epi_any'], A['epi_ge2'], A['epi_ge6'] = (epi_o >= 1).astype(float).where(epi_o.notna()), (epi_o >= 2).astype(float).where(epi_o.notna()), (epi_o >= 3).astype(float).where(epi_o.notna())
diam_o = ordinal(diam, [(r'0 to', 1), (r'2 to', 2), (r'>\s*6', 3)])
A['infil_ge2'], A['infil_ge6'] = (diam_o >= 2).astype(float).where(diam_o.notna()), (diam_o >= 3).astype(float).where(diam_o.notna())
A['depth_mid'] = np.where(blank(depth), np.nan, depth.str.contains('middle|posterior|endothel').astype(float))
A['depth_post'] = np.where(blank(depth), np.nan, depth.str.contains('posterior|endothel').astype(float))
A['loc_central'] = np.where(blank(loc), np.nan, loc.str.contains('central|diffuse').astype(float))
A['loc_peripheral'] = np.where(blank(loc), np.nan, loc.str.contains('peripheral|diffuse').astype(float))
A['loc_diffuse'] = np.where(blank(loc), np.nan, loc.str.contains('diffuse').astype(float))
A['limbus'] = np.where(blank(limb), np.nan, limb.str.startswith('yes').astype(float))
for name, flag in [('feathery', 'feathery'), ('satellite', 'satellite'), ('ring', 'ring infiltrate'), ('endothelial_plaque', 'endothelial plaque'),
                   ('dry_surface', 'dry ulcer'), ('pigmented', 'pigmented'), ('superficial_plaque', 'superficial plaque'), ('kp', 'keratic')]:
    A[name] = yes_no(spec, flag)
A['perforation'] = pd.concat([yes_no(spec, 'impending perforation'), yes_no(spec, 'frank perforation')], axis=1).max(axis=1)
AUX_COLS = [f'aux_{c}' for c in A.columns]
A.columns = AUX_COLS; A['key'] = ex['key'].values
m = m.merge(A, on='key', how='left')
cov = m[m.present][AUX_COLS].notna().mean().round(2); prev = m[m.present][AUX_COLS].mean().round(3)
print('aux targets (rows with a record): coverage / prevalence'); print(pd.DataFrame({'coverage': cov, 'prevalence': prev}).T.to_string())
print('\nprevalence by label (rows with a record):'); print(m[m.present].groupby('label')[AUX_COLS].mean().round(2).T.to_string())

# ---- blue-light pairs: same (patient, visit, eye); same device preferred; nearest in time ----
root_cache = 's3_root_listing.csv'
blue = raw[raw['is_blue'] & ~raw['non_eye']][['patient_id', 'visit_id', 'eye_l', 'device_n', 'img_date', 'id']].copy()
blue['visit_id'] = blue['visit_id'].astype(str)
blue = blue.rename(columns={'id': 'blue_id', 'device_n': 'blue_device', 'img_date': 'blue_date'})
cand = m[['id', 'patient_id', 'visit_id', 'eye_l', 'device_n', 'img_date']].merge(blue, on=['patient_id', 'visit_id', 'eye_l'])
cand['same_dev'] = (cand['device_n'] == cand['blue_device']).astype(int)
cand['dt'] = (cand['img_date'] - cand['blue_date']).abs().dt.total_seconds().fillna(1e12)
cand = cand.sort_values(['id', 'same_dev', 'dt'], ascending=[True, False, True]).drop_duplicates('id')
m['blue_id'] = m['id'].map(cand.set_index('id')['blue_id'])
print(f'\nblue-light pairs: {m.blue_id.notna().sum():,} of {len(m):,} labelled diffuse rows have a blue image of the same eye-visit '
      f'({m[m.blue_id.notna()].device_n.value_counts().to_dict()})')

# ---- visit order ----
m['visit_date'] = m.groupby(['patient_id', 'visit_id'])['img_date'].transform('min')
m['visit_rank'] = m.groupby('patient_id')['visit_date'].rank(method='dense')
m['is_index'] = (m['visit_rank'] == 1).astype(int)
print(f'visits per patient: {m.groupby("patient_id")["visit_id"].nunique().describe()[["mean", "50%", "max"]].round(2).to_dict()}; '
      f'index-visit rows: {int(m.is_index.sum()):,}')

aux targets (rows with a record): coverage / prevalence
            aux_hypopyon  aux_hypopyon_ge2mm  aux_epi_any  aux_epi_ge2  aux_epi_ge6  aux_infil_ge2  aux_infil_ge6  aux_depth_mid  aux_depth_post  aux_loc_central  aux_loc_peripheral  aux_loc_diffuse  aux_limbus  aux_feathery  aux_satellite  aux_ring  aux_endothelial_plaque  aux_dry_surface  aux_pigmented  aux_superficial_plaque  aux_kp  aux_perforation
coverage           0.990               0.970        0.990        0.990        0.990          0.920          0.920          0.990           0.990            0.990               0.990            0.990       0.990         0.990          0.990     0.990                    0.99            0.990          0.990                   0.990   0.990            0.990
prevalence         0.255               0.103        0.902        0.675        0.149          0.739          0.163          0.596           0.255            0.541               0.658            0.069       0.159         0.299          

## 2. Build the dataset

Diffuse images and splits come from `v4-diffuse-1024`; the paired blue-light images are staged into the same prefix at 1024 px (resumable, low-memory). Manifest rows: **train / val / test** = `infected`, `infected_propagated`, `single_unmatched` only (ambiguous rows made no difference in v6 and are left out). Every row carries `blue_image`, `has_blue`, `is_index`, the `meta_*` features and the `aux_*` targets.

In [5]:
from PIL import Image, ImageFile, ImageDraw
ImageFile.LOAD_TRUNCATED_IMAGES = True

def s3_exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key); return True
    except Exception:
        return False

def list_prefix(prefix):
    names = set()
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=BUCKET, Prefix=f'{prefix}/'):
        for o in page.get('Contents', []):
            names.add(o['Key'].split('/')[-1])
    return names

def list_root_images(cache='s3_root_listing.csv'):
    if os.path.exists(cache):
        d = pd.read_csv(cache, dtype=str); return dict(zip(d['id'], d['key']))
    mp = {}
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=BUCKET):
        for o in page.get('Contents', []):
            k = o['Key']
            if '/' in k: continue
            base = k[:-4] if k.endswith('.jpg') else k
            if base not in mp or k.endswith('.jpg'): mp[base] = k
    pd.DataFrame({'id': list(mp), 'key': list(mp.values())}).to_csv(cache, index=False)
    return mp

def resize_bytes(raw_bytes, short_side):
    img = Image.open(io.BytesIO(raw_bytes)); img.draft('RGB', (short_side, short_side)); img = img.convert('RGB')
    w, h = img.size; scale = short_side / min(w, h)
    if scale < 1:
        img = img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.BILINEAR)
    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=JPEG_QUALITY)
    return buf.getvalue()

def stage_images(names, out_prefix, short_side, workers=3, chunk=100, sidecar='staged.csv'):
    """Resize originals (bucket root) to `short_side` and upload under out_prefix/. Resumable through a sidecar list in the prefix."""
    side_key, side_local = f'{out_prefix}/{sidecar}', f'{out_prefix}_{sidecar}'
    staged = set()
    if s3_exists(side_key):
        s3.download_file(BUCKET, side_key, side_local); staged = set(pd.read_csv(side_local, dtype=str)['image_name'])
    done = list_prefix(out_prefix)
    todo = [n for n in dict.fromkeys(names) if n not in done]
    print(f'  staging -> {out_prefix}/ @ {short_side}px: {len(done):,} present, {len(todo):,} to process')
    def one(name):
        try:
            s3.put_object(Bucket=BUCKET, Key=f'{out_prefix}/{name}', Body=resize_bytes(s3.get_object(Bucket=BUCKET, Key=name)['Body'].read(), short_side))
            return name, True
        except Exception:
            return name, False
    failed, t0 = [], time.time()
    for start in range(0, len(todo), chunk):
        with ThreadPoolExecutor(max_workers=workers) as ex_:
            for name, ok in ex_.map(one, todo[start:start + chunk]):
                (staged.add if ok else failed.append)(name)
        gc.collect()
        if (start // chunk) % 10 == 9 or start + chunk >= len(todo):
            pd.DataFrame({'image_name': sorted(staged | done)}).to_csv(side_local, index=False); s3.upload_file(side_local, BUCKET, side_key)
            el = (time.time() - t0) / 60; n = min(start + chunk, len(todo))
            print(f'    {n:,}/{len(todo):,}  failed {len(failed)}  {el:.1f} min  eta {el / n * (len(todo) - n):.0f} min', flush=True)
    return (staged | done), failed

def assign_splits(df, prior_manifests, seed=SPLIT_SEED):
    df['split'] = np.nan
    for pm in prior_manifests:
        inh = pm.assign(patient_id=pid_str(pm['patient_id'])).groupby('patient_id')['split'].first()
        df['split'] = df['split'].fillna(df['patient_id'].map(inh))
    new_p = df.loc[df['split'].isna(), ['patient_id', 'label']].drop_duplicates('patient_id')
    if 0 < len(new_p) < 10:
        df['split'] = df['split'].fillna('train')
    elif len(new_p):
        from sklearn.model_selection import train_test_split
        strat = new_p['label'] if new_p['label'].value_counts().min() >= 4 else None
        tv, te = train_test_split(new_p, test_size=0.15, stratify=strat, random_state=seed)
        tr, va = train_test_split(tv, test_size=0.15 / 0.85, stratify=(tv['label'] if strat is not None else None), random_state=seed)
        assign = {**{p: 'train' for p in tr.patient_id}, **{p: 'val' for p in va.patient_id}, **{p: 'test' for p in te.patient_id}}
        df['split'] = df['split'].fillna(df['patient_id'].map(assign))
    print(f'  splits: {df.patient_id.nunique() - len(new_p):,} patients inherited, {len(new_p):,} newly assigned')
    return df

SYM_SCALE, VA_LO, VA_HI = np.log1p(365.0), -0.3, 3.0
def add_meta_features(df):
    keep = ['image_name', 'blue_image', 'has_blue', 'label', 'patient_id', 'split', 'eye', 'device', 'site', 'visit_id', 'visit_rank', 'is_index',
            'determined', 'eye_status', 'clin_source', 'clin_sym_days', 'clin_va'] + AUX_COLS
    out = df[keep].copy()
    out['eye'] = norm(out['eye']).str.lower(); out['device'] = norm(out['device']).str.lower(); out['site'] = norm(out['site']).str.upper()
    month, day = df['month'], pd.to_numeric(df['created_at'].astype(str).str.slice(8, 10), errors='coerce')
    out['meta_month_sin'], out['meta_month_cos'] = np.sin(2*np.pi*month/12).fillna(0), np.cos(2*np.pi*month/12).fillna(0)
    out['meta_day_sin'], out['meta_day_cos'] = np.sin(2*np.pi*day/31).fillna(0), np.cos(2*np.pi*day/31).fillna(0)
    out['meta_date_missing'] = month.isna().astype(float)
    train_mask = df['split'] == 'train'
    cats = {'eye': out['eye'], 'device': out['device'], 'site': out['site'], 'season': month.map(SEASONS).fillna('unknown')}
    for col, v in cats.items():
        v = v.replace({'nan': 'missing'}); kp = v[train_mask].value_counts(); kp = kp[kp >= 20].index
        v = v.where(v.isin(kp), 'other')
        if v[train_mask].nunique() < 2: continue
        out = pd.concat([out, pd.get_dummies(v, prefix=f'meta_{col}').astype(float)], axis=1)
    sym = pd.to_numeric(df['clin_sym_days'], errors='coerce').clip(0, 365)
    out['meta_clin_sym_days'] = (np.log1p(sym) / SYM_SCALE).fillna(0.0); out['meta_clin_sym_days_missing'] = sym.isna().astype(float)
    va = pd.to_numeric(df['clin_va'], errors='coerce').clip(VA_LO, VA_HI)
    out['meta_clin_va_logmar'] = ((va - VA_LO) / (VA_HI - VA_LO)).fillna(0.0); out['meta_clin_va_missing'] = va.isna().astype(float)
    rc = df['clin_risk']
    for c in RF_CATS:
        out[f'meta_clin_rf_{c}'] = rc.map(lambda v: float(isinstance(v, list) and c in v)).astype(float)
    out['meta_clin_rf_missing'] = rc.map(lambda v: float(not isinstance(v, list))).astype(float)
    return out

def pair_qa(mf, n=8, path='pair_qa_v7.png'):
    s = mf[mf.has_blue == 1].sample(min(n, int(mf.has_blue.sum())), random_state=1); tiles = []
    for r in s.itertuples():
        try:
            a = Image.open(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=f'{DATA_PREFIX}/{r.image_name}')['Body'].read())).convert('RGB')
            b = Image.open(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=f'{DATA_PREFIX}/{r.blue_image}')['Body'].read())).convert('RGB')
        except Exception:
            continue
        for im in (a, b): im.thumbnail((300, 300))
        t = Image.new('RGB', (600, 300), 'white'); t.paste(a, (0, 0)); t.paste(b, (300, 0))
        ImageDraw.Draw(t).text((4, 4), f'{r.device[:5]} {r.label[:1]} {r.eye}', fill=(255, 255, 0)); tiles.append(t)
    if not tiles: return
    grid = Image.new('RGB', (1200, 300 * ((len(tiles) + 1) // 2)), 'white')
    for i, t in enumerate(tiles): grid.paste(t, ((i % 2) * 600, (i // 2) * 300))
    grid.save(path); print(f'pair QA grid -> {path}')

def build_dataset():
    if not FORCE_REBUILD and s3_exists(f'{DATA_PREFIX}/{V7_ALL}') and s3_exists(f'{DATA_PREFIX}/{V7_LABELS}'):
        s3.download_file(BUCKET, f'{DATA_PREFIX}/{V7_ALL}', V7_ALL); mf = pd.read_csv(V7_ALL); mf['patient_id'] = pid_str(mf['patient_id'])
        print(f'{V7_LABELS}: manifest already in S3 -- reusing it'); return mf
    print(f'{V7_LABELS}: building')
    root = list_root_images()
    df = m.copy()
    df['image_name'] = df['id'].map(root); df['blue_image'] = df['blue_id'].map(root).fillna('')
    n0 = len(df); df = df[df['image_name'].notna()]
    conflict = df.groupby('patient_id')['label'].nunique(); bad = conflict[conflict > 1].index
    df = df[~df['patient_id'].isin(bad)].drop_duplicates('image_name').reset_index(drop=True)
    df = df.drop(columns=[c for c in ('device', 'site') if c in df.columns]).rename(columns={'device_n': 'device', 'site_n': 'site'})
    print(f'  {n0:,} rows -> {len(df):,} with an image; dropped {len(bad)} label-conflict patients -> {df.patient_id.nunique():,} patients')
    priors = []
    for p in (V4_PREFIX, V3_PREFIX, BASE_PREFIX):
        if s3_exists(f'{p}/{LABELS_FILE}'):
            s3.download_file(BUCKET, f'{p}/{LABELS_FILE}', f'{p}_{LABELS_FILE}'); priors.append(pd.read_csv(f'{p}_{LABELS_FILE}'))
    df = assign_splits(df, priors)
    keep_rows = df['eye_status'].isin(CLEAN_STATUS)
    staged, failed = stage_images(list(df.loc[keep_rows, 'image_name']) + [b for b in df.loc[keep_rows, 'blue_image'] if b], DATA_PREFIX, TARGET_SHORT_SIDE)
    df = df[df['image_name'].isin(staged) | ~keep_rows].reset_index(drop=True)
    df.loc[~df['blue_image'].isin(staged), 'blue_image'] = ''
    df['has_blue'] = (df['blue_image'] != '').astype(int)
    mf = add_meta_features(df)
    assert (mf.groupby('patient_id')['split'].nunique() == 1).all()
    by_split = mf.groupby('split')['patient_id'].agg(set)
    assert not (by_split['train'] & by_split['test']) and not (by_split['train'] & by_split['val'])
    mf.to_csv(V7_ALL, index=False); s3.upload_file(V7_ALL, BUCKET, f'{DATA_PREFIX}/{V7_ALL}')
    keep = mf['eye_status'].isin(CLEAN_STATUS)
    mf[keep].to_csv(V7_LABELS, index=False); s3.upload_file(V7_LABELS, BUCKET, f'{DATA_PREFIX}/{V7_LABELS}')
    print(f'  uploaded {V7_LABELS} ({int(keep.sum()):,} rows) and {V7_ALL} ({len(mf):,} rows; {len(failed)} images failed to stage)')
    return mf

MAN7 = build_dataset()
USE7 = MAN7[MAN7['eye_status'].isin(CLEAN_STATUS)]
AUX_COLS = [c for c in MAN7.columns if c.startswith('aux_')]
print(f'\n{V7_LABELS}: {len(USE7):,} rows / {USE7.patient_id.nunique():,} patients; {int(USE7.has_blue.sum()):,} with a blue pair; '
      f'{len([c for c in MAN7.columns if c.startswith("meta_")])} meta columns, {len(AUX_COLS)} aux targets')
t = USE7.pivot_table(index='split', columns='label', values='image_name', aggfunc='size', fill_value=0)
t['patients'] = USE7.groupby('split')['patient_id'].nunique(); t['eye_visits'] = USE7.groupby('split').apply(lambda g: g.groupby(['patient_id', 'visit_id', 'eye']).ngroups)
t['index_visits'] = USE7[USE7.is_index == 1].groupby('split').apply(lambda g: g.groupby(['patient_id', 'visit_id', 'eye']).ngroups); t['with_blue'] = USE7.groupby('split')['has_blue'].sum()
print(t.reindex(['train', 'val', 'test']).to_string())
print('\ntest eye-visits by device x has_blue:'); tt = USE7[USE7.split == 'test'].drop_duplicates(['patient_id', 'visit_id', 'eye']); print(pd.crosstab(tt['device'], tt['has_blue']).to_string())
pair_qa(USE7[USE7.split == 'test'])

labels_v7.csv: building
  40,789 rows -> 22,647 with an image; dropped 13 label-conflict patients -> 5,007 patients
  splits: 5,007 patients inherited, 0 newly assigned
  staging -> v4-diffuse-1024/ @ 1024px: 23,353 present, 8,193 to process
    1,000/8,193  failed 0  2.5 min  eta 18 min
    2,000/8,193  failed 0  4.7 min  eta 15 min
    3,000/8,193  failed 0  6.9 min  eta 12 min
    4,000/8,193  failed 0  9.1 min  eta 10 min
    5,000/8,193  failed 0  11.2 min  eta 7 min
    6,000/8,193  failed 0  13.7 min  eta 5 min
    7,000/8,193  failed 0  16.1 min  eta 3 min
    8,000/8,193  failed 0  18.4 min  eta 0 min
    8,193/8,193  failed 0  18.8 min  eta 0 min
  uploaded labels_v7.csv (9,746 rows) and labels_v7_all.csv (22,647 rows; 0 images failed to stage)

labels_v7.csv: 9,746 rows / 3,097 patients; 9,341 with a blue pair; 32 meta columns, 22 aux targets
label  Bacterial  Fungal  Viral  patients  eye_visits  index_visits  with_blue
split                                                  

## 3. Training scripts

In [6]:
%%writefile src/requirements.txt
timm>=0.9.12,<2


Overwriting src/requirements.txt


In [7]:
%%writefile src/train.py
"""
Keratitis image classifier -- SageMaker training entry point (v10: paired views, auxiliary morphology, distillation,
hallucination, visit-level metrics, full-pool cross-validation).

One script, three tasks, selected with --task:
  bf     Bacterial vs Fungal                 (binary, positive = Fungal)
  bfv    Bacterial vs Fungal vs Viral        (3-class, macro one-vs-rest AUROC)
  tier1  Viral vs non-viral (B or F)         (binary, positive = Viral)

Rows of the manifest whose label is outside the task get y = -1: never trained on, still predicted.

THE EVALUATION UNIT (v10). The manifest carries eye-level labels (column 'eye_status'); val/test rows are images of the
diseased eye only. Metrics are reported at four levels:
  image         every test image
  visit         one score per (patient, visit, eye): mean of that eye-visit's images     <- MAIN METRIC and selection default
  index_visit   visit level restricted to each patient's first labelled visit            <- the deployment scenario
  patient       mean over all of a patient's images (cross-visit average, kept for continuity with v3-v6)
Confidence intervals always resample patients, so correlated visits do not shrink them.

v10 options (all off by default):
  --views pair              feed the paired blue-light image (manifest column 'blue_image') through the SAME backbone and
                            concatenate [f_diffuse, f_blue, meta]; a learned null vector stands in when no pair exists.
  --aux-weight W            auxiliary morphology heads on the manifest columns 'aux_*' (0/1/NaN; NaN = unknown, masked).
                            Train-time only: they regularise the backbone towards the features clinicians use.
  --kd-weight W             knowledge distillation from a teacher's exported outputs (channel 'teacher': model.tar.gz with
                            train_outputs.npz); KL(student || teacher) at temperature --kd-temp on the training rows.
  --halluc-weight W         modality hallucination (Hoffman et al. 2016): a second backbone maps the DIFFUSE image to the
                            teacher's BLUE feature (cosine loss on rows that have a pair); the head sees [f_d, f_hallucinated, meta].
                            Needs only the diffuse image at inference.
  --init-backbone PATH      initialise the backbone(s) from a self-supervised checkpoint (channel 'pretrained': backbone.pth
                            or model.tar.gz containing it).
  --export-outputs 1        after training, write train_outputs.npz (logits, probs, diffuse and blue features for every
                            train/val/test row) so the run can serve as a teacher.
  --fold k --n-folds K --cv-pool all
                            K-fold over ALL patients (train+val+test pooled): fold k is the test set, a grouped --cv-val-frac
                            slice of the rest is the validation set. Pool out-of-fold test predictions across the K runs.

Artifacts in SM_MODEL_DIR: best_model.pth, model.pth, config.json, performance_summary.json, history.csv,
  val_predictions.csv, test_predictions.csv, {val,test}_visit_predictions.csv, {val,test}_patient_predictions.csv,
  train_outputs.npz (if --export-outputs), unreadable_images.txt (only if any).

Local run:  python train.py --data-dir ./data --model-dir ./out --task bf --arch resnet18 --pretrained 0 --img-size 64 --epochs 1 --workers 0
"""
import argparse
import copy
import glob
import json
import math
import os
import random
import tarfile
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False

ImageFile.LOAD_TRUNCATED_IMAGES = True
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

TASKS = {
    "bf":    {"classes": ["Bacterial", "Fungal"],          "map": {"Bacterial": 0, "Fungal": 1}},
    "bfv":   {"classes": ["Bacterial", "Fungal", "Viral"], "map": {"Bacterial": 0, "Fungal": 1, "Viral": 2}},
    "tier1": {"classes": ["NonViral", "Viral"],            "map": {"Bacterial": 0, "Fungal": 0, "Viral": 1}},
}
EXTRA_COLS = ["site", "device", "eye", "determined", "visit_id", "eye_status", "is_index", "has_blue"]   # carried into prediction files
GROUP_COLS = ("site", "device", "determined", "eye_status", "has_blue")


def log(*a):
    print(*a, flush=True)


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


# --------------------------------------------------------------------------------------
# Artifacts (teacher outputs, pretrained backbone)
# --------------------------------------------------------------------------------------
def find_artifact(path, member, scratch):
    """path = a file, a directory holding `member`, or a directory/file model.tar.gz containing `member`. Returns a local file path."""
    if not path:
        return None
    if os.path.isfile(path) and path.endswith(member):
        return path
    cands = [path] if os.path.isfile(path) else sorted(glob.glob(os.path.join(path, "**", "*"), recursive=True))
    for c in cands:
        if os.path.isfile(c) and os.path.basename(c) == member:
            return c
    for c in cands:
        if os.path.isfile(c) and c.endswith((".tar.gz", ".tgz", ".tar")):
            with tarfile.open(c) as t:
                for m in t.getmembers():
                    if os.path.basename(m.name) == member:
                        os.makedirs(scratch, exist_ok=True)
                        t.extract(m, scratch)
                        return os.path.join(scratch, m.name)
    raise FileNotFoundError(f"{member} not found under {path}")


# --------------------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------------------
def build_backbone(arch, img_size, pretrained=True):
    """Return (backbone, feat_dim, mean, std); backbone maps (B,3,H,W) -> (B, feat_dim)."""
    mean, std = IMAGENET_MEAN, IMAGENET_STD
    tv = {
        "resnet18": (models.resnet18, "IMAGENET1K_V1"),
        "resnet50": (models.resnet50, "IMAGENET1K_V2"),
        "convnext_tiny": (models.convnext_tiny, "IMAGENET1K_V1"),
        "convnext_small": (models.convnext_small, "IMAGENET1K_V1"),
        "efficientnet_v2_s": (models.efficientnet_v2_s, "IMAGENET1K_V1"),
        "efficientnet_v2_m": (models.efficientnet_v2_m, "IMAGENET1K_V1"),
    }
    if arch in tv:
        ctor, w = tv[arch]
        m = ctor(weights=w if pretrained else None)
        if arch.startswith("resnet"):
            d, m.fc = m.fc.in_features, nn.Identity()
        elif arch.startswith("convnext"):
            d = m.classifier[2].in_features
            m.classifier = nn.Sequential(m.classifier[0], m.classifier[1])
        else:
            d, m.classifier = m.classifier[1].in_features, nn.Identity()
        return m, d, mean, std
    if arch.startswith("timm:"):
        if not HAS_TIMM:
            raise RuntimeError("timm not installed -- add 'timm' to src/requirements.txt")
        name = arch[len("timm:"):]
        kwargs = dict(pretrained=pretrained, num_classes=0)
        try:
            m = timm.create_model(name, img_size=img_size, **kwargs)
        except TypeError:
            m = timm.create_model(name, **kwargs)
        cfg = timm.data.resolve_data_config({}, model=m)
        return m, m.num_features, tuple(cfg["mean"]), tuple(cfg["std"])
    raise ValueError(f"unknown arch {arch}")


def load_backbone_weights(module, path, tag):
    sd = torch.load(path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    sd = {k[len("backbone."):] if k.startswith("backbone.") else k: v for k, v in sd.items()}
    sd = {k: v for k, v in sd.items() if not k.startswith(("heads.", "meta_mlp.", "aux_head.", "halluc_net.", "null_blue", "proj."))}
    res = module.load_state_dict(sd, strict=False)
    log(f"[{tag}] loaded {len(sd) - len(res.unexpected_keys)} tensors from {os.path.basename(path)}; "
        f"missing {len(res.missing_keys)}, unexpected {len(res.unexpected_keys)}")


class Net(nn.Module):
    """Backbone -> [f_diffuse | f_blue or f_hallucinated | metadata MLP] -> head(s) (+ auxiliary morphology head)."""

    def __init__(self, arch, img_size, n_classes, dropout, n_meta=0, meta_dim=32, meta_drop=0.1, head_hidden=256,
                 pretrained=True, n_domains=1, views="diffuse", halluc=False, n_aux=0):
        super().__init__()
        self.backbone, d, self.mean, self.std = build_backbone(arch, img_size, pretrained)
        self.d, self.n_meta, self.meta_drop, self.n_domains, self.views, self.halluc = d, n_meta, meta_drop, n_domains, views, halluc
        fused = d
        if views == "pair":
            self.null_blue = nn.Parameter(torch.zeros(d))
            fused += d
        elif halluc:
            self.halluc_net, _, _, _ = build_backbone(arch, img_size, pretrained)
            fused += d
        if n_meta > 0:
            self.meta_mlp = nn.Sequential(nn.Linear(n_meta, 64), nn.ReLU(inplace=True), nn.Dropout(dropout),
                                          nn.Linear(64, meta_dim), nn.ReLU(inplace=True))
            fused += meta_dim
        else:
            self.meta_mlp = None

        def make_head():
            if head_hidden > 0:
                return nn.Sequential(nn.Dropout(dropout), nn.Linear(fused, head_hidden), nn.ReLU(inplace=True),
                                     nn.Dropout(dropout), nn.Linear(head_hidden, n_classes))
            return nn.Sequential(nn.Dropout(dropout), nn.Linear(fused, n_classes))
        self.heads = nn.ModuleList([make_head() for _ in range(max(1, n_domains))])
        self.aux_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(fused, n_aux)) if n_aux > 0 else None

    def forward(self, b, return_all=False):
        f = self.backbone(b["img"])
        parts, extras = [f], {"f_d": f}
        if self.views == "pair":
            fb = self.backbone(b["blue"])
            has = b["has_blue"].view(-1, 1).bool()
            extras["f_b_raw"] = fb
            fb = torch.where(has, fb, self.null_blue.to(fb.dtype).unsqueeze(0).expand_as(fb))
            parts.append(fb)
        elif self.halluc:
            fh = self.halluc_net(b["img"])
            extras["f_h"] = fh
            parts.append(fh)
        if self.n_meta > 0:
            meta = b["meta"]
            if self.training and self.meta_drop > 0:
                keep = (torch.rand(meta.shape[0], 1, device=meta.device) >= self.meta_drop).float()
                meta = meta * keep
            parts.append(self.meta_mlp(meta))
        fused = torch.cat(parts, dim=1)
        if self.n_domains <= 1:
            logits = self.heads[0](fused)
        else:
            outs = torch.stack([h(fused) for h in self.heads], dim=1)                 # (B, D, C)
            logits = outs.gather(1, b["dom"].view(-1, 1, 1).expand(-1, 1, outs.shape[2])).squeeze(1)
        if self.aux_head is not None:
            extras["aux"] = self.aux_head(fused)
        return (logits, extras) if return_all else logits

    def backbone_params(self):
        p = list(self.backbone.parameters())
        if self.halluc:
            p += list(self.halluc_net.parameters())
        return p

    def head_params(self):
        p = list(self.heads.parameters())
        if self.meta_mlp is not None:
            p += list(self.meta_mlp.parameters())
        if self.aux_head is not None:
            p += list(self.aux_head.parameters())
        if self.views == "pair":
            p += [self.null_blue]
        return p


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


# --------------------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------------------
def build_transforms(S, eval_resize, mean, std, aug):
    train = [transforms.RandomResizedCrop(S, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
             transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(p=0.2),
             transforms.RandomRotation(20), transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
             transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1)]
    if aug == "strong":
        train.append(transforms.TrivialAugmentWide())
    train += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    if aug == "strong":
        train.append(transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)))
    evalt = [transforms.Resize(eval_resize), transforms.CenterCrop(S), transforms.ToTensor(), transforms.Normalize(mean, std)]
    return transforms.Compose(train), transforms.Compose(evalt)


def grayworld(img):
    a = np.asarray(img).astype(np.float32)
    m = a.reshape(-1, 3).mean(0) + 1e-6
    a = a * (m.mean() / m)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))


def load_img(path, transform, crop_frac, box=None, color_norm="none", crop_pad=1.15):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    if box is not None and box[2] > 0:
        cx, cy, side = box[0] * w, box[1] * h, box[2] * min(w, h) * crop_pad
        half = side / 2
        l, t = int(max(0, min(cx - half, w - side))), int(max(0, min(cy - half, h - side)))
        r_, b_ = int(min(w, l + side)), int(min(h, t + side))
        if r_ - l > 32 and b_ - t > 32:
            img = img.crop((l, t, r_, b_))
    elif crop_frac < 1.0:
        side = int(min(w, h) * crop_frac)
        l, t = (w - side) // 2, (h - side) // 2
        img = img.crop((l, t, l + side, t + side))
    if color_norm == "grayworld":
        img = grayworld(img)
    return transform(img)


def validate_images(names, data_dir, workers=16):
    from concurrent.futures import ThreadPoolExecutor

    def ok(name):
        try:
            with Image.open(os.path.join(data_dir, name)) as im:
                im.verify()
            return True
        except Exception:
            return False

    with ThreadPoolExecutor(max_workers=workers) as ex:
        return np.array(list(ex.map(ok, list(names))), dtype=bool)


class ImageDataset(Dataset):
    """Returns a dict: img, blue, has_blue, meta, y, idx, dom, aux, aux_m, t_logits, t_m, t_fb, t_fbm.
    train=True: a corrupt diffuse file is replaced by another training image; train=False: zero image for the same row."""

    def __init__(self, df, data_dir, transform, crop_frac, meta_cols, img_size, train, crop_mode="none", color_norm="none",
                 crop_pad=1.15, views="diffuse", blue_col="blue_image", aux_cols=(), teacher=None, n_classes=2, feat_dim=1):
        self.df = df.reset_index(drop=True)
        n = len(self.df)
        self.data_dir, self.transform, self.crop_frac, self.img_size, self.train = data_dir, transform, crop_frac, img_size, train
        self.color_norm, self.crop_pad, self.views = color_norm, crop_pad, views
        self.meta = self.df[meta_cols].values.astype("float32") if meta_cols else np.zeros((n, 0), dtype="float32")
        self.y = self.df["y"].values.astype("int64")
        self.dom = self.df["dom"].values.astype("int64") if "dom" in self.df.columns else np.zeros(n, dtype="int64")
        self.boxes = (self.df[["cx", "cy", "side"]].values.astype("float32")
                      if crop_mode == "manifest" and {"cx", "cy", "side"} <= set(self.df.columns) else None)
        self.blue = (self.df[blue_col].fillna("").astype(str).values if (views == "pair" and blue_col in self.df.columns)
                     else np.array([""] * n))
        aux = self.df[list(aux_cols)].values.astype("float32") if aux_cols else np.zeros((n, 0), dtype="float32")
        self.aux_m = (~np.isnan(aux)).astype("float32")
        self.aux = np.nan_to_num(aux).astype("float32")
        # teacher outputs aligned to this frame
        self.t_logits, self.t_m = np.zeros((n, n_classes), "float32"), np.zeros(n, "float32")
        self.t_fb, self.t_fbm = np.zeros((n, feat_dim), "float32"), np.zeros(n, "float32")
        if teacher is not None:
            pos = pd.Series(np.arange(len(teacher["image_name"])), index=teacher["image_name"])
            j = pos.reindex(self.df["image_name"].values).values
            ok = ~pd.isna(j)
            jj = j[ok].astype(int)
            if teacher["logits"].shape[1] == n_classes:
                self.t_logits[ok] = teacher["logits"][jj]
                self.t_m[ok] = 1.0
            if "feat_b" in teacher and teacher["feat_b"].shape[1] == feat_dim:
                fb = teacher["feat_b"][jj]
                good = ~np.isnan(fb).any(1)
                idx = np.where(ok)[0][good]
                self.t_fb[idx] = fb[good]
                self.t_fbm[idx] = 1.0

    def __len__(self):
        return len(self.df)

    def _blue(self, idx):
        name = self.blue[idx]
        if self.views != "pair":
            return torch.zeros(1), 0.0
        if not name or name == "nan":
            return torch.zeros(3, self.img_size, self.img_size), 0.0
        try:
            return load_img(os.path.join(self.data_dir, name), self.transform, 1.0, None, self.color_norm), 1.0
        except Exception as e:
            log(f"WARNING: failed to load blue {name} ({e})")
            return torch.zeros(3, self.img_size, self.img_size), 0.0

    def _row(self, idx, img):
        blue, has = self._blue(idx)
        return {"img": img, "blue": blue, "has_blue": torch.tensor(has, dtype=torch.float32), "meta": torch.from_numpy(self.meta[idx]),
                "y": int(self.y[idx]), "idx": idx, "dom": int(self.dom[idx]), "aux": torch.from_numpy(self.aux[idx]),
                "aux_m": torch.from_numpy(self.aux_m[idx]), "t_logits": torch.from_numpy(self.t_logits[idx]),
                "t_m": torch.tensor(self.t_m[idx]), "t_fb": torch.from_numpy(self.t_fb[idx]), "t_fbm": torch.tensor(self.t_fbm[idx])}

    def __getitem__(self, idx):
        orig = idx
        for _ in range(10):
            try:
                box = tuple(self.boxes[idx]) if self.boxes is not None else None
                img = load_img(os.path.join(self.data_dir, self.df.at[idx, "image_name"]), self.transform, self.crop_frac,
                               box, self.color_norm, self.crop_pad)
                return self._row(idx, img)
            except Exception as e:
                log(f"WARNING: failed to load {self.df.at[idx, 'image_name']} ({e})")
                if not self.train:
                    return self._row(orig, torch.zeros(3, self.img_size, self.img_size))
                idx = random.randrange(len(self.df))
        raise RuntimeError("too many image load failures")


def grouped_folds(pat, n_folds, seed):
    try:
        from sklearn.model_selection import StratifiedGroupKFold
        return list(StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed).split(pat, pat["label"], groups=pat["patient_id"]))
    except ImportError:
        from sklearn.model_selection import GroupKFold
        return list(GroupKFold(n_splits=n_folds).split(pat, pat["label"], groups=pat["patient_id"]))


def make_splits(df, args):
    """Fixed split column; or K-fold over train+val (test untouched); or --cv-pool all: K-fold over every patient,
    fold k = test, a grouped --cv-val-frac slice of the remainder = val."""
    if args.fold < 0:
        return (df[df["split"] == "train"].reset_index(drop=True), df[df["split"] == "val"].reset_index(drop=True),
                df[df["split"] == "test"].reset_index(drop=True))
    if args.cv_pool == "all":
        pool = df.reset_index(drop=True)
        pat = pool.groupby("patient_id")["label"].first().reset_index()
        folds = grouped_folds(pat, args.n_folds, args.split_seed)
        test_pids = set(pat.iloc[folds[args.fold][1]]["patient_id"])
        rest = pat[~pat["patient_id"].isin(test_pids)].reset_index(drop=True)
        n_val_folds = max(2, int(round(1 / args.cv_val_frac)))
        vf = grouped_folds(rest, n_val_folds, args.split_seed + 1 + args.fold)
        val_pids = set(rest.iloc[vf[0][1]]["patient_id"])
        return (pool[~pool["patient_id"].isin(test_pids | val_pids)].reset_index(drop=True),
                pool[pool["patient_id"].isin(val_pids)].reset_index(drop=True),
                pool[pool["patient_id"].isin(test_pids)].reset_index(drop=True))
    test_df = df[df["split"] == "test"].reset_index(drop=True)
    pool = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
    pat = pool.groupby("patient_id")["label"].first().reset_index()
    folds = grouped_folds(pat, args.n_folds, args.split_seed)
    val_pids = set(pat.iloc[folds[args.fold][1]]["patient_id"])
    return (pool[~pool["patient_id"].isin(val_pids)].reset_index(drop=True),
            pool[pool["patient_id"].isin(val_pids)].reset_index(drop=True), test_df)


# --------------------------------------------------------------------------------------
# Losses / metrics
# --------------------------------------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight, self.gamma, self.ls = weight, gamma, label_smoothing

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight, label_smoothing=self.ls, reduction="none")
        pt = torch.exp(-F.cross_entropy(logits, target, reduction="none"))
        return ((1 - pt) ** self.gamma * ce).mean()


def masked_bce(logits, target, mask):
    if mask.sum() == 0:
        return logits.sum() * 0.0
    l = F.binary_cross_entropy_with_logits(logits.float(), target, reduction="none")
    return (l * mask).sum() / mask.sum()


def kd_loss(s_logits, t_logits, mask, T):
    if mask.sum() == 0:
        return s_logits.sum() * 0.0
    l = F.kl_div(F.log_softmax(s_logits.float() / T, dim=1), F.softmax(t_logits / T, dim=1), reduction="none").sum(1)
    return (l * mask).sum() / mask.sum() * (T * T)


def halluc_loss(f_h, t_fb, mask):
    if mask.sum() == 0:
        return f_h.sum() * 0.0
    l = 1.0 - F.cosine_similarity(f_h.float(), t_fb, dim=1)
    return (l * mask).sum() / mask.sum()


def auroc(y, P):
    y, P = np.asarray(y), np.asarray(P)
    if P.shape[1] == 2:
        return roc_auc_score(y, P[:, 1]) if len(np.unique(y)) > 1 else float("nan")
    per = per_class_auroc(y, P)
    vals = [v for v in per if not np.isnan(v)]
    return float(np.mean(vals)) if vals else float("nan")


def per_class_auroc(y, P):
    y, P = np.asarray(y), np.asarray(P)
    out = []
    for c in range(P.shape[1]):
        pos = (y == c)
        out.append(float(roc_auc_score(pos, P[:, c])) if 0 < pos.sum() < len(y) else float("nan"))
    return out


def cluster_boot_ci(y, P, groups, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    y, P, groups = np.asarray(y), np.asarray(P), np.asarray(groups)
    uniq = np.unique(groups)
    idx_by = {g: np.where(groups == g)[0] for g in uniq}
    vals = []
    for _ in range(n_boot):
        i = np.concatenate([idx_by[g] for g in rng.choice(uniq, size=len(uniq), replace=True)])
        a = auroc(y[i], P[i])
        if not np.isnan(a):
            vals.append(a)
    if len(vals) < 50:
        return float("nan"), float("nan")
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[int(np.argmax(tpr - fpr))])


def report(y, P, groups, classes, tag, thr=None):
    y, P = np.asarray(y), np.asarray(P)
    if len(y) == 0:
        return {"n": 0, "auroc": float("nan")}
    binary = P.shape[1] == 2
    if binary:
        thr = 0.5 if thr is None else thr
        pred = (P[:, 1] >= thr).astype(int)
    else:
        pred = P.argmax(1)
    auc = auroc(y, P)
    lo, hi = cluster_boot_ci(y, P, groups)
    acc = accuracy_score(y, pred)
    cm = confusion_matrix(y, pred, labels=list(range(len(classes))))
    out = {"n": int(len(y)), "n_patients": int(len(np.unique(groups))), "auroc": float(auc), "auroc_ci95": [lo, hi], "accuracy": float(acc),
           "confusion_matrix": cm.tolist(), "class_support": [int((y == c).sum()) for c in range(len(classes))]}
    if binary:
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=1, zero_division=0)
        out.update(threshold=float(thr), precision=float(pr), recall=float(rc), f1=float(f1))
        log(f"[{tag}] n={len(y)} ({out['n_patients']} patients) thr={thr:.3f}  AUROC {auc:.4f} [{lo:.4f}, {hi:.4f}]  acc {acc:.4f}  "
            f"prec {pr:.4f}  rec {rc:.4f}  F1 {f1:.4f}")
    else:
        pcs = per_class_auroc(y, P)
        out["per_class_auroc"] = dict(zip(classes, pcs))
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
        out.update(macro_precision=float(pr), macro_recall=float(rc), macro_f1=float(f1))
        log(f"[{tag}] n={len(y)} ({out['n_patients']} patients)  macro-OvR AUROC {auc:.4f} [{lo:.4f}, {hi:.4f}]  acc {acc:.4f}  "
            f"macro-F1 {f1:.4f}  " + "  ".join(f"{c}:{v:.3f}" for c, v in zip(classes, pcs)))
    return out


def to_dev(b, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}


@torch.no_grad()
def predict(model, loader, device, tta, use_amp, amp_dtype, feats=False):
    model.eval()
    probs, logits, ys, order, fd, fb, hasb = [], [], [], [], [], [], []
    for b in loader:
        b = to_dev(b, device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            lg, ex = model(b, return_all=True)
            p = torch.softmax(lg.float(), dim=1)
            if tta:
                bf = dict(b)
                bf["img"] = torch.flip(b["img"], dims=[3])
                if b["blue"].dim() == 4:
                    bf["blue"] = torch.flip(b["blue"], dims=[3])
                p = (p + torch.softmax(model(bf).float(), dim=1)) / 2
        probs.append(p.cpu().numpy())
        logits.append(lg.float().cpu().numpy())
        ys.extend(b["y"].cpu().numpy())
        order.extend(b["idx"].cpu().numpy())
        if feats:
            fd.append(ex["f_d"].float().cpu().numpy())
            if "f_b_raw" in ex:
                fb.append(ex["f_b_raw"].float().cpu().numpy())
                hasb.append(b["has_blue"].cpu().numpy())
    out = {"y": np.array(ys), "P": np.concatenate(probs, 0), "order": np.array(order), "logits": np.concatenate(logits, 0)}
    if feats:
        out["f_d"] = np.concatenate(fd, 0)
        if fb:
            f = np.concatenate(fb, 0)
            f[np.concatenate(hasb, 0) < 0.5] = np.nan
            out["f_b"] = f
    return out


def mode_or_first(s):
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]


def rollup(df, pcols, how="mean", by="patient_id"):
    """One row per group: label/y, n images, mean (or median) of each prob column, majority of the group columns."""
    keys = [by] if isinstance(by, str) else list(by)
    agg = {**{c: (c, how) for c in pcols}, "y": ("y", "first"), "label": ("label", "first"), "n": ("y", "size")}
    for c in GROUP_COLS + ("is_index", "patient_id"):
        if c in df.columns and c not in keys:
            agg[c] = (c, mode_or_first)
    return df.groupby(keys).agg(**agg).reset_index()


def visit_keys(df):
    keys = ["patient_id"]
    if "visit_id" in df.columns:
        keys.append("visit_id")
    if "eye" in df.columns:
        keys.append("eye")
    return keys


def levels(df, pcols, how="mean"):
    """{'image': df, 'visit': per (patient, visit, eye), 'index_visit': visits flagged is_index, 'patient': per patient}."""
    v = rollup(df, pcols, how, by=visit_keys(df))
    out = {"image": df, "visit": v, "patient": rollup(df, pcols, how, by="patient_id")}
    if "is_index" in v.columns:
        out["index_visit"] = v[pd.to_numeric(v["is_index"], errors="coerce").fillna(0).astype(int) == 1].reset_index(drop=True)
    return out


# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------
def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default=os.environ.get("SM_CHANNEL_TRAINING", "./data"))
    ap.add_argument("--model-dir", default=os.environ.get("SM_MODEL_DIR", "./model"))
    ap.add_argument("--labels-file", default="labels_v7.csv")
    ap.add_argument("--task", default="bf", choices=sorted(TASKS))
    # data
    ap.add_argument("--use-meta", type=int, default=1)
    ap.add_argument("--meta-set", default="all", choices=["all", "basic"])
    ap.add_argument("--meta-drop", type=float, default=0.1)
    ap.add_argument("--fold", type=int, default=-1)
    ap.add_argument("--n-folds", type=int, default=5)
    ap.add_argument("--cv-pool", default="trainval", choices=["trainval", "all"])
    ap.add_argument("--cv-val-frac", type=float, default=0.15)
    ap.add_argument("--split-seed", type=int, default=42)
    ap.add_argument("--sampler", default="none", choices=["none", "patient", "class"])
    ap.add_argument("--train-rows", default="all", choices=["all", "determined", "matched"])
    ap.add_argument("--ambiguous", default="drop", choices=["drop", "keep"])
    ap.add_argument("--train-device", default="all")
    ap.add_argument("--device-heads", type=int, default=0)
    ap.add_argument("--crop-mode", default="none", choices=["none", "manifest"])
    ap.add_argument("--crop-pad", type=float, default=1.15)
    ap.add_argument("--color-norm", default="none", choices=["none", "grayworld"])
    ap.add_argument("--max-train-images", type=int, default=0)
    ap.add_argument("--validate-images", type=int, default=1)
    # v10
    ap.add_argument("--views", default="diffuse", choices=["diffuse", "pair"])
    ap.add_argument("--blue-col", default="blue_image")
    ap.add_argument("--aux-weight", type=float, default=0.0)
    ap.add_argument("--aux-prefix", default="aux_")
    ap.add_argument("--kd-weight", type=float, default=0.0)
    ap.add_argument("--kd-temp", type=float, default=2.0)
    ap.add_argument("--halluc-weight", type=float, default=0.0)
    ap.add_argument("--halluc-init", default="teacher", choices=["teacher", "same"],
                    help="teacher = init the hallucination backbone from the teacher's backbone weights; same = like the main backbone")
    ap.add_argument("--teacher-dir", default=os.environ.get("SM_CHANNEL_TEACHER", ""))
    ap.add_argument("--init-backbone", default=os.environ.get("SM_CHANNEL_PRETRAINED", ""))
    ap.add_argument("--export-outputs", type=int, default=0)
    # model
    ap.add_argument("--arch", default="efficientnet_v2_s")
    ap.add_argument("--pretrained", type=int, default=1)
    ap.add_argument("--img-size", type=int, default=448)
    ap.add_argument("--eval-resize", type=int, default=0)
    ap.add_argument("--crop-frac", type=float, default=1.0)
    ap.add_argument("--dropout", type=float, default=0.5)
    ap.add_argument("--head-hidden", type=int, default=256)
    ap.add_argument("--grad-ckpt", type=int, default=0)
    # optimisation
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--grad-accum", type=int, default=2)
    ap.add_argument("--head-lr", type=float, default=3e-4)
    ap.add_argument("--backbone-lr-mult", type=float, default=0.1)
    ap.add_argument("--weight-decay", type=float, default=1e-2)
    ap.add_argument("--warmup-epochs", type=float, default=1.0)
    ap.add_argument("--freeze-epochs", type=int, default=0)
    ap.add_argument("--clip-grad", type=float, default=0.0)
    ap.add_argument("--ema", type=int, default=1)
    ap.add_argument("--ema-decay", type=float, default=0.999)
    ap.add_argument("--amp", type=int, default=1)
    ap.add_argument("--bf16", type=int, default=1)
    # regularisation / objective
    ap.add_argument("--aug", default="light", choices=["light", "strong"])
    ap.add_argument("--loss", default="ce", choices=["ce", "focal"])
    ap.add_argument("--focal-gamma", type=float, default=2.0)
    ap.add_argument("--label-smoothing", type=float, default=0.05)
    ap.add_argument("--cw-power", type=float, default=1.0)
    # selection / eval
    ap.add_argument("--select-on", default="visit", choices=["image", "visit", "patient", "mean"])
    ap.add_argument("--patience", type=int, default=6)
    ap.add_argument("--time-budget-min", type=float, default=0)
    ap.add_argument("--tta", type=int, default=1)
    ap.add_argument("--rollup", default="mean", choices=["mean", "median"])
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--seed", type=int, default=42)
    return ap.parse_args()


def main():
    args = parse_args()
    set_seed(args.seed)
    os.makedirs(args.model_dir, exist_ok=True)
    scratch = os.path.join(args.model_dir, "_inputs")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = bool(args.amp) and device.type == "cuda"
    amp_dtype = torch.bfloat16 if (use_amp and args.bf16 and torch.cuda.is_bf16_supported()) else torch.float16
    torch.backends.cudnn.benchmark = True
    t0 = time.time()
    task = TASKS[args.task]
    classes, n_classes = task["classes"], len(task["classes"])
    pcols = [f"prob_{c}" for c in classes]
    halluc = args.halluc_weight > 0
    if halluc and args.views == "pair":
        raise ValueError("--halluc-weight needs --views diffuse (the point is to not need the blue image)")

    # ---- data ----
    df = pd.read_csv(os.path.join(args.data_dir, args.labels_file))
    df["patient_id"] = df["patient_id"].astype(str)
    df["y"] = df["label"].map(task["map"]).fillna(-1).astype(int)
    if "eye_status" not in df.columns:
        df["eye_status"] = "unknown"
    df["eye_status"] = df["eye_status"].astype(str)
    if args.views == "pair" and args.blue_col in df.columns:
        df["has_blue"] = (df[args.blue_col].fillna("").astype(str).str.len() > 0).astype(int)
    elif "has_blue" not in df.columns:
        df["has_blue"] = 0
    if args.validate_images:
        tv = time.time()
        readable = validate_images(df["image_name"], args.data_dir)
        bad = df.loc[~readable, "image_name"].tolist()
        log(f"image validation: {len(bad)} unreadable of {len(df)} ({time.time() - tv:.0f}s)")
        if bad:
            with open(os.path.join(args.model_dir, "unreadable_images.txt"), "w") as f:
                f.write("\n".join(bad))
            df = df[readable].reset_index(drop=True)
        if args.views == "pair" and args.blue_col in df.columns:
            bl = df[args.blue_col].fillna("").astype(str)
            rb = validate_images(bl[bl.str.len() > 0], args.data_dir)
            nbad = int((~rb).sum())
            if nbad:
                log(f"blue-image validation: {nbad} unreadable pairs dropped (rows keep their diffuse image)")
                df.loc[bl[bl.str.len() > 0].index[~rb], args.blue_col] = ""
                df["has_blue"] = (df[args.blue_col].fillna("").astype(str).str.len() > 0).astype(int)
    meta_cols = sorted(c for c in df.columns if c.startswith("meta_")) if args.use_meta else []
    if args.meta_set == "basic":
        meta_cols = [c for c in meta_cols if not c.startswith("meta_clin_")]
    aux_cols = sorted(c for c in df.columns if c.startswith(args.aux_prefix)) if args.aux_weight > 0 else []
    devices = sorted(df["device"].astype(str).str.lower().unique()) if "device" in df.columns else ["all"]
    dom_map = {d: i for i, d in enumerate(devices)}
    df["dom"] = df["device"].astype(str).str.lower().map(dom_map).fillna(0).astype(int) if "device" in df.columns else 0
    n_domains = len(devices) if args.device_heads else 1
    train_all, val_all, test_all = make_splits(df, args)
    train_df = train_all[train_all["y"] >= 0].reset_index(drop=True)
    val_df = val_all[val_all["y"] >= 0].reset_index(drop=True)
    if args.train_rows == "determined" and "determined" in df.columns:
        keep = (train_df["determined"].fillna(0).astype(int) == 1) | (train_df["label"] == "Viral")
        train_df = train_df[keep].reset_index(drop=True)
    if args.train_rows == "matched":
        train_df = train_df[train_df["eye_status"] == "infected"].reset_index(drop=True)
    if args.ambiguous == "drop":
        train_df = train_df[train_df["eye_status"] != "ambiguous"].reset_index(drop=True)
    val_df = val_df[val_df["eye_status"] != "ambiguous"].reset_index(drop=True)
    if args.train_device != "all" and "device" in df.columns:
        m = lambda d: d["device"].astype(str).str.lower() == args.train_device.lower()
        train_df, val_df = train_df[m(train_df)].reset_index(drop=True), val_df[m(val_df)].reset_index(drop=True)
    if args.max_train_images and len(train_df) > args.max_train_images:
        train_df = train_df.sample(args.max_train_images, random_state=args.seed).reset_index(drop=True)
    assert not set(train_df.patient_id) & set(val_all.patient_id), "patient leakage train/val"
    assert not set(train_df.patient_id) & set(test_all.patient_id), "patient leakage train/test"
    assert train_df["y"].nunique() == n_classes, f"train split is missing a class: {train_df['label'].value_counts().to_dict()}"

    # ---- teacher outputs ----
    teacher = None
    if args.kd_weight > 0 or halluc:
        tp = find_artifact(args.teacher_dir, "train_outputs.npz", scratch)
        z = np.load(tp, allow_pickle=True)
        teacher = {k: z[k] for k in z.files}
        teacher["image_name"] = teacher["image_name"].astype(str)
        log(f"teacher outputs: {len(teacher['image_name'])} rows, logits {teacher['logits'].shape}, "
            f"blue features for {int((~np.isnan(teacher['feat_b']).any(1)).sum()) if 'feat_b' in teacher else 0} rows")

    log(f"task={args.task} classes={classes} arch={args.arch} img={args.img_size} meta={len(meta_cols)} ({args.meta_set}) aux={len(aux_cols)} "
        f"views={args.views} halluc={halluc} kd={args.kd_weight} fold={args.fold}/{args.n_folds} pool={args.cv_pool} device={device} "
        f"amp={use_amp}/{str(amp_dtype).split('.')[-1]} timm={HAS_TIMM}")
    for nm, d in [("train", train_df), ("val", val_df), ("test(in-task)", test_all[test_all["y"] >= 0])]:
        log(f"{nm}: {len(d)} images, {d['patient_id'].nunique()} patients, {d['label'].value_counts().to_dict()}"
            + (f", with blue pair {int(d['has_blue'].sum())}" if args.views == "pair" else ""))
    log(f"val rows predicted (all labels): {len(val_all)} | test rows predicted (all labels): {len(test_all)}")

    model = Net(args.arch, args.img_size, n_classes, args.dropout, n_meta=len(meta_cols), meta_drop=args.meta_drop,
                head_hidden=args.head_hidden, pretrained=bool(args.pretrained), n_domains=n_domains, views=args.views,
                halluc=halluc, n_aux=len(aux_cols))
    if args.init_backbone:
        bp = find_artifact(args.init_backbone, "backbone.pth", scratch)
        load_backbone_weights(model.backbone, bp, "backbone <- pretrained")
        if halluc and args.halluc_init == "same":
            load_backbone_weights(model.halluc_net, bp, "halluc_net <- pretrained")
    if halluc and args.halluc_init == "teacher":
        try:
            tb = find_artifact(args.teacher_dir, "best_model.pth", scratch)
            load_backbone_weights(model.halluc_net, tb, "halluc_net <- teacher backbone")
        except FileNotFoundError:
            log("teacher best_model.pth not found; hallucination backbone keeps its default init")
    model = model.to(device)
    if args.grad_ckpt and hasattr(model.backbone, "set_grad_checkpointing"):
        model.backbone.set_grad_checkpointing(True)
    n_all = sum(p.numel() for p in model.parameters())
    log(f"total params {n_all / 1e6:.1f}M  feat_dim={model.d}  norm mean={model.mean} std={model.std}")

    S = args.img_size
    eval_resize = args.eval_resize or int(round(S * 256 / 224))
    train_tf, eval_tf = build_transforms(S, eval_resize, model.mean, model.std, args.aug)
    dlk = dict(num_workers=args.workers, pin_memory=device.type == "cuda")
    if args.workers > 0:
        dlk.update(persistent_workers=True, prefetch_factor=4)
    ds_kw = dict(crop_mode=args.crop_mode, color_norm=args.color_norm, crop_pad=args.crop_pad, views=args.views, blue_col=args.blue_col,
                 aux_cols=aux_cols, teacher=teacher, n_classes=n_classes, feat_dim=model.d)

    def make_train_loader(d):
        ds = ImageDataset(d, args.data_dir, train_tf, args.crop_frac, meta_cols, S, train=True, **ds_kw)
        if args.sampler != "none":
            w = (1.0 / d.groupby("patient_id")["image_name"].transform("size").values if args.sampler == "patient"
                 else 1.0 / d.groupby("y")["image_name"].transform("size").values)
            sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(ds), replacement=True)
            return DataLoader(ds, batch_size=args.batch_size, sampler=sampler, drop_last=True, **dlk)
        return DataLoader(ds, batch_size=args.batch_size, shuffle=True, drop_last=True, **dlk)

    train_loader = make_train_loader(train_df)
    mk_eval = lambda d: DataLoader(ImageDataset(d, args.data_dir, eval_tf, args.crop_frac, meta_cols, S, train=False, **ds_kw),
                                   batch_size=args.batch_size, shuffle=False, **dlk)
    val_loader, val_all_loader, test_loader = mk_eval(val_df), mk_eval(val_all), mk_eval(test_all)
    if teacher is not None:
        tds = train_loader.dataset
        log(f"teacher coverage of training rows: logits {int(tds.t_m.sum())}/{len(tds)}, blue features {int(tds.t_fbm.sum())}/{len(tds)}")
    if aux_cols:
        cov = train_loader.dataset.aux_m.mean(0)
        log("aux targets (train coverage): " + ", ".join(f"{c[len(args.aux_prefix):]}={v:.2f}" for c, v in zip(aux_cols, cov)))

    # ---- objective ----
    cw = None
    if args.cw_power > 0:
        counts = train_df["y"].value_counts().sort_index().reindex(range(n_classes)).fillna(1).values.astype(float)
        w = (1.0 / counts) ** args.cw_power
        w = w / w.mean()
        cw = torch.tensor(w, dtype=torch.float32, device=device)
        log("class weights " + str(dict(zip(classes, np.round(w, 3)))))
    criterion = (FocalLoss(cw, args.focal_gamma, args.label_smoothing) if args.loss == "focal"
                 else nn.CrossEntropyLoss(weight=cw, label_smoothing=args.label_smoothing))

    def groups(params, lr):
        return [{"params": [p for p in params if p.ndim > 1], "lr": lr, "weight_decay": args.weight_decay},
                {"params": [p for p in params if p.ndim <= 1], "lr": lr, "weight_decay": 0.0}]

    optimizer = torch.optim.AdamW(groups(model.backbone_params(), args.head_lr * args.backbone_lr_mult)
                                  + groups(model.head_params(), args.head_lr))
    spe = max(1, len(train_loader) // args.grad_accum)
    total_steps, warm = args.epochs * spe, max(1, int(args.warmup_epochs * spe))

    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        prog = (step - warm) / max(1, total_steps - warm)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16)
    ema = ModelEMA(model, args.ema_decay) if args.ema else None
    eval_model = ema.module if ema else model

    def val_scores(m):
        r = predict(m, val_loader, device, False, use_amp, amp_dtype)
        vd = val_df.iloc[r["order"]].reset_index(drop=True).copy()
        for j, c in enumerate(pcols):
            vd[c] = r["P"][:, j]
        lv = levels(vd, pcols, args.rollup)
        sc = {k: auroc(v["y"].values, v[pcols].values) for k, v in lv.items()}
        sc["mean"] = (sc["image"] + sc["patient"]) / 2
        return sc

    # ---- train ----
    best, best_epoch, best_state, no_imp, history = -1.0, -1, None, 0, []
    for epoch in range(args.epochs):
        frozen = epoch < args.freeze_epochs
        for p in model.backbone_params():
            p.requires_grad_(not frozen)
        model.train()
        tot = {"loss": 0.0, "ce": 0.0, "aux": 0.0, "kd": 0.0, "hal": 0.0}
        nb, te = 0, time.time()
        optimizer.zero_grad(set_to_none=True)
        for i, b in enumerate(train_loader):
            b = to_dev(b, device)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                out, ex = model(b, return_all=True)
                ce = criterion(out, b["y"])
                loss = ce
                if aux_cols:
                    la = masked_bce(ex["aux"], b["aux"], b["aux_m"])
                    loss = loss + args.aux_weight * la
                    tot["aux"] += la.item()
                if args.kd_weight > 0:
                    lk = kd_loss(out, b["t_logits"], b["t_m"], args.kd_temp)
                    loss = loss + args.kd_weight * lk
                    tot["kd"] += lk.item()
                if halluc:
                    lh = halluc_loss(ex["f_h"], b["t_fb"], b["t_fbm"])
                    loss = loss + args.halluc_weight * lh
                    tot["hal"] += lh.item()
                loss = loss / args.grad_accum
            scaler.scale(loss).backward()
            if (i + 1) % args.grad_accum == 0:
                if args.clip_grad > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_grad)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                if ema:
                    ema.update(model)
            tot["loss"] += loss.item() * args.grad_accum
            tot["ce"] += ce.item()
            nb += 1
        tot = {k: v / max(1, nb) for k, v in tot.items()}

        sc = val_scores(eval_model)
        sel = sc[args.select_on]
        lr_now = optimizer.param_groups[2]["lr"]
        history.append({"epoch": epoch + 1, "train_loss": tot["loss"], "train_ce": tot["ce"], "train_aux": tot["aux"], "train_kd": tot["kd"],
                        "train_halluc": tot["hal"], "val_image_auroc": sc["image"], "val_visit_auroc": sc["visit"],
                        "val_index_auroc": sc.get("index_visit", float("nan")), "val_patient_auroc": sc["patient"], "lr": lr_now,
                        "frozen_backbone": frozen, "epoch_sec": time.time() - te, "n_train_rows": int(len(train_df))})
        log(f"Epoch {epoch + 1}/{args.epochs} - loss {tot['loss']:.4f} (ce {tot['ce']:.3f}" + (f" aux {tot['aux']:.3f}" if aux_cols else "")
            + (f" kd {tot['kd']:.3f}" if args.kd_weight > 0 else "") + (f" hal {tot['hal']:.3f}" if halluc else "")
            + f") - val image {sc['image']:.4f} visit {sc['visit']:.4f} index {sc.get('index_visit', float('nan')):.4f} "
            f"patient {sc['patient']:.4f} - lr {lr_now:.2e} - {time.time() - te:.0f}s")
        if not np.isnan(sel) and sel > best + 1e-4:
            best, best_epoch, no_imp = sel, epoch + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in eval_model.state_dict().items()}
            torch.save(best_state, os.path.join(args.model_dir, "best_model.pth"))
            log(f"  new best val {args.select_on} AUROC {best:.4f} - saved")
        else:
            no_imp += 1
            log(f"  no improvement ({no_imp}/{args.patience})")
            if no_imp >= args.patience:
                log(f"early stopping at epoch {epoch + 1}")
                break
        if args.time_budget_min and (time.time() - t0) / 60 > args.time_budget_min:
            log(f"time budget of {args.time_budget_min:.0f} min reached after epoch {epoch + 1}; stopping training")
            break

    # ---- final evaluation with the best checkpoint; predictions for ALL val/test rows ----
    if best_state is not None:
        eval_model.load_state_dict(best_state)
    tta = bool(args.tta)

    def predict_all(loader, src_df, name):
        r = predict(eval_model, loader, device, tta, use_amp, amp_dtype)
        keep_cols = ["image_name", "patient_id", "label", "y"] + [c for c in EXTRA_COLS if c in src_df.columns]
        d = src_df.iloc[r["order"]].reset_index(drop=True)[keep_cols].copy()
        for j, c in enumerate(pcols):
            d[c] = r["P"][:, j]
        d.to_csv(os.path.join(args.model_dir, f"{name}_predictions.csv"), index=False)
        lv = levels(d, pcols, args.rollup)
        lv["visit"].to_csv(os.path.join(args.model_dir, f"{name}_visit_predictions.csv"), index=False)
        lv["patient"].to_csv(os.path.join(args.model_dir, f"{name}_patient_predictions.csv"), index=False)
        return d, lv

    vd, vlv = predict_all(val_all_loader, val_all, "val")
    td, tlv = predict_all(test_loader, test_all, "test")
    intask = lambda f: f[f["y"] >= 0]
    vin, tin = intask(vd), intask(td)
    binary = n_classes == 2
    thr_img = youden_threshold(vin["y"], vin[pcols[1]]) if binary else None
    val_sc = {k: auroc(intask(v)["y"].values, intask(v)[pcols].values) for k, v in vlv.items()}
    log(f"val (best ckpt, tta={tta}): " + "  ".join(f"{k} {v:.4f}" for k, v in val_sc.items()))

    log("=" * 70)
    log(f"TEST RESULTS - task={args.task} {args.arch} views={args.views} halluc={halluc} aux={len(aux_cols)} kd={args.kd_weight} @ {S}px "
        f"best_epoch={best_epoch} (val {args.select_on} AUROC {best:.4f})")
    summary = {"task": args.task, "classes": classes, "arch": args.arch, "img_size": S, "use_meta": int(bool(meta_cols)), "meta_cols": meta_cols,
               "aux_cols": aux_cols, "views": args.views, "halluc": halluc, "fold": args.fold, "n_folds": args.n_folds, "cv_pool": args.cv_pool,
               "best_epoch": best_epoch, "best_val_select_auroc": float(best), "select_on": args.select_on, "n_params_total": int(n_all),
               "val_image_auroc": float(val_sc["image"]), "val_visit_auroc": float(val_sc["visit"]), "val_patient_auroc": float(val_sc["patient"]),
               "val_index_auroc": float(val_sc.get("index_visit", float("nan"))), "thr_youden_image": thr_img,
               "n_train": int(len(train_df)), "n_val": int(len(val_df)), "n_test": int(len(tin)),
               "n_test_with_blue": int(tin["has_blue"].sum()) if "has_blue" in tin.columns else 0,
               "train_class_counts": train_df["label"].value_counts().to_dict(), "test_class_counts": tin["label"].value_counts().to_dict(),
               "test_patient_ids": sorted(tin["patient_id"].unique().tolist()), "hyperparameters": vars(args), "history": history,
               "train_minutes": (time.time() - t0) / 60}
    for lvl, key in [("image", "image_level"), ("visit", "visit_level"), ("index_visit", "index_visit_level"), ("patient", "patient_level")]:
        if lvl in tlv:
            f = intask(tlv[lvl])
            summary[key] = report(f["y"], f[pcols].values, f["patient_id"].values, classes, lvl)
    if binary:
        summary["image_level_youden"] = report(tin["y"], tin[pcols].values, tin["patient_id"].values, classes, "image @youden", thr_img)
    # paired-subset metrics (test eye-visits that have a blue-light pair), for teacher-vs-student comparisons on identical rows
    if "has_blue" in tin.columns and tin["has_blue"].sum() > 0:
        pv = intask(levels(tin[tin["has_blue"] == 1], pcols, args.rollup)["visit"])
        summary["visit_level_paired_subset"] = report(pv["y"], pv[pcols].values, pv["patient_id"].values, classes, "visit, rows with a blue pair")
    # stratified metrics (image + visit level, no CI)
    summary["by_group"] = {}
    for col in GROUP_COLS:
        if col in tin.columns and tin[col].nunique() > 1:
            summary["by_group"][col] = {}
            for val, g in tin.groupby(tin[col].astype(str)):
                vv = levels(g, pcols, args.rollup)["visit"]
                summary["by_group"][col][val] = {"n_img": int(len(g)), "n_visit": int(len(vv)), "image_auroc": float(auroc(g["y"], g[pcols].values)),
                                                 "visit_auroc": float(auroc(vv["y"], vv[pcols].values))}
                log(f"[{col}={val}] n={len(g)} imgs/{len(vv)} eye-visits  image AUROC {summary['by_group'][col][val]['image_auroc']:.4f}  "
                    f"visit AUROC {summary['by_group'][col][val]['visit_auroc']:.4f}")
    # auxiliary heads: what did the backbone learn about morphology? (test AUROC per aux target)
    if aux_cols:
        ds = test_loader.dataset
        la = []
        with torch.no_grad():
            for b in test_loader:
                b = to_dev(b, device)
                with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                    la.append(model_aux_logits(eval_model, b).float().cpu().numpy())
        la = np.concatenate(la, 0)
        summary["aux_test_auroc"] = {}
        for j, c in enumerate(aux_cols):
            msk = ds.aux_m[:, j] > 0
            yy = ds.aux[msk, j]
            summary["aux_test_auroc"][c[len(args.aux_prefix):]] = (float(roc_auc_score(yy, la[msk, j])) if 0 < yy.sum() < len(yy) else float("nan"))
        log("aux test AUROC: " + ", ".join(f"{k}={v:.3f}" for k, v in summary["aux_test_auroc"].items()))
    log("=" * 70)

    # ---- teacher export: outputs for every train/val/test row ----
    if args.export_outputs:
        te0 = time.time()
        names, logits, probs, fd, fb = [], [], [], [], []
        for d in (train_all, val_all, test_all):
            if len(d) == 0:
                continue
            r = predict(eval_model, mk_eval(d), device, False, use_amp, amp_dtype, feats=True)
            names.append(d.iloc[r["order"]]["image_name"].values.astype(str))
            logits.append(r["logits"]); probs.append(r["P"]); fd.append(r["f_d"])
            fb.append(r.get("f_b", np.full((len(r["order"]), model.d), np.nan, dtype="float32")))
        np.savez_compressed(os.path.join(args.model_dir, "train_outputs.npz"), image_name=np.concatenate(names), logits=np.concatenate(logits),
                            probs=np.concatenate(probs), feat_d=np.concatenate(fd).astype("float32"), feat_b=np.concatenate(fb).astype("float32"),
                            classes=np.array(classes))
        log(f"exported train_outputs.npz for {sum(len(n) for n in names)} rows ({time.time() - te0:.0f}s)")

    torch.save(eval_model.state_dict(), os.path.join(args.model_dir, "model.pth"))
    pd.DataFrame(history).to_csv(os.path.join(args.model_dir, "history.csv"), index=False)
    with open(os.path.join(args.model_dir, "config.json"), "w") as f:
        json.dump(vars(args), f, indent=2)
    with open(os.path.join(args.model_dir, "performance_summary.json"), "w") as f:
        json.dump(summary, f, indent=2, default=float)
    log(f"done in {(time.time() - t0) / 60:.1f} min")


@torch.no_grad()
def model_aux_logits(model, b):
    model.eval()
    _, ex = model(b, return_all=True)
    return ex["aux"]


if __name__ == "__main__":
    main()


Overwriting src/train.py


In [8]:
%%writefile src/pretrain.py
"""
Self-supervised pretraining on the whole image archive (SimCLR / NT-Xent) with cross-view positives.

Input channel 'training' holds the archive images and a manifest CSV (default archive_manifest.csv) with columns
  image_name   file in the channel
  pair         (optional) file name of the paired view of the same eye-visit (diffuse <-> blue light), '' if none
For a sample with a pair, the second view is an augmentation of the PAIRED image with probability --pair-prob (Contrastive
Multiview Coding); otherwise both views are augmentations of the same image (SimCLR). Colour augmentation is kept mild
because colour is diagnostic for this task.

Output: backbone.pth (torchvision/timm backbone state dict, loadable by train.py --init-backbone) + pretrain_log.csv.
"""
import argparse
import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from train import build_backbone, log, set_seed, validate_images

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PairDataset(Dataset):
    def __init__(self, df, data_dir, tf, pair_prob):
        self.names = df["image_name"].astype(str).values
        self.pairs = df["pair"].fillna("").astype(str).values if "pair" in df.columns else np.array([""] * len(df))
        self.data_dir, self.tf, self.pair_prob = data_dir, tf, pair_prob

    def __len__(self):
        return len(self.names)

    def _open(self, name):
        return Image.open(os.path.join(self.data_dir, name)).convert("RGB")

    def __getitem__(self, i):
        for _ in range(10):
            try:
                img = self._open(self.names[i])
                other = img
                cross = 0.0
                if self.pairs[i] and random.random() < self.pair_prob:
                    try:
                        other = self._open(self.pairs[i])
                        cross = 1.0
                    except Exception:
                        other = img
                return self.tf(img), self.tf(other), torch.tensor(cross)
            except Exception as e:
                log(f"WARNING: failed to load {self.names[i]} ({e})")
                i = random.randrange(len(self.names))
        raise RuntimeError("too many image load failures")


class SimCLR(nn.Module):
    def __init__(self, arch, img_size, pretrained, proj_dim):
        super().__init__()
        self.backbone, d, self.mean, self.std = build_backbone(arch, img_size, pretrained)
        self.proj = nn.Sequential(nn.Linear(d, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Linear(512, proj_dim))

    def forward(self, x):
        return F.normalize(self.proj(self.backbone(x)).float(), dim=1)


def nt_xent(z1, z2, temp):
    z = torch.cat([z1, z2], 0)
    n = z1.shape[0]
    sim = z @ z.t() / temp
    sim.fill_diagonal_(-1e9)
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default=os.environ.get("SM_CHANNEL_TRAINING", "./archive"))
    ap.add_argument("--model-dir", default=os.environ.get("SM_MODEL_DIR", "./pretrain_out"))
    ap.add_argument("--manifest", default="archive_manifest.csv")
    ap.add_argument("--arch", default="efficientnet_v2_s")
    ap.add_argument("--pretrained", type=int, default=1, help="start from ImageNet weights (1) or random (0)")
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--weight-decay", type=float, default=1e-4)
    ap.add_argument("--warmup-epochs", type=float, default=2.0)
    ap.add_argument("--temp", type=float, default=0.1)
    ap.add_argument("--proj-dim", type=int, default=128)
    ap.add_argument("--pair-prob", type=float, default=0.5)
    ap.add_argument("--max-images", type=int, default=0)
    ap.add_argument("--validate-images", type=int, default=1)
    ap.add_argument("--time-budget-min", type=float, default=0)
    ap.add_argument("--workers", type=int, default=12)
    ap.add_argument("--amp", type=int, default=1)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    set_seed(args.seed)
    os.makedirs(args.model_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = bool(args.amp) and device.type == "cuda"
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    torch.backends.cudnn.benchmark = True
    t0 = time.time()

    df = pd.read_csv(os.path.join(args.data_dir, args.manifest))
    if args.max_images and len(df) > args.max_images:
        df = df.sample(args.max_images, random_state=args.seed).reset_index(drop=True)
    if args.validate_images:
        ok = validate_images(df["image_name"], args.data_dir)
        log(f"image validation: {int((~ok).sum())} unreadable of {len(df)}")
        df = df[ok].reset_index(drop=True)
    n_pairs = int((df["pair"].fillna("").astype(str).str.len() > 0).sum()) if "pair" in df.columns else 0
    log(f"archive: {len(df)} images, {n_pairs} with a paired view ({args.pair_prob:.0%} cross-view positives)")

    model = SimCLR(args.arch, args.img_size, bool(args.pretrained), args.proj_dim).to(device)
    S = args.img_size
    tf = transforms.Compose([
        transforms.RandomResizedCrop(S, scale=(0.25, 1.0), ratio=(0.8, 1.25)),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(p=0.2), transforms.RandomRotation(30),
        transforms.RandomApply([transforms.ColorJitter(0.3, 0.3, 0.2, 0.02)], p=0.6),
        transforms.RandomGrayscale(p=0.05),
        transforms.RandomApply([transforms.GaussianBlur(23, sigma=(0.1, 2.0))], p=0.3),
        transforms.ToTensor(), transforms.Normalize(model.mean, model.std)])
    ds = PairDataset(df, args.data_dir, tf, args.pair_prob)
    dlk = dict(num_workers=args.workers, pin_memory=device.type == "cuda", drop_last=True, shuffle=True)
    if args.workers > 0:
        dlk.update(persistent_workers=True, prefetch_factor=4)
    loader = DataLoader(ds, batch_size=args.batch_size, **dlk)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    spe = len(loader)
    total, warm = args.epochs * spe, max(1, int(args.warmup_epochs * spe))
    sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16)
    log(f"arch={args.arch} img={S} batch={args.batch_size} steps/epoch={spe} device={device} amp={use_amp}/{str(amp_dtype).split('.')[-1]}")

    rows = []
    for epoch in range(args.epochs):
        model.train()
        tot, ncross, nb, te = 0.0, 0.0, 0, time.time()
        for x1, x2, cross in loader:
            x1, x2 = x1.to(device, non_blocking=True), x2.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                z = model(torch.cat([x1, x2], 0))
                loss = nt_xent(z[: x1.shape[0]], z[x1.shape[0]:], args.temp)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            sched.step()
            tot += loss.item(); ncross += cross.sum().item(); nb += 1
        rows.append({"epoch": epoch + 1, "loss": tot / max(1, nb), "cross_view_share": ncross / max(1, nb * args.batch_size),
                     "lr": optimizer.param_groups[0]["lr"], "epoch_sec": time.time() - te})
        log(f"Epoch {epoch + 1}/{args.epochs} - NT-Xent {rows[-1]['loss']:.4f} - cross-view share {rows[-1]['cross_view_share']:.2f} - "
            f"lr {rows[-1]['lr']:.2e} - {rows[-1]['epoch_sec']:.0f}s")
        torch.save(model.backbone.state_dict(), os.path.join(args.model_dir, "backbone.pth"))
        pd.DataFrame(rows).to_csv(os.path.join(args.model_dir, "pretrain_log.csv"), index=False)
        if args.time_budget_min and (time.time() - t0) / 60 > args.time_budget_min:
            log(f"time budget of {args.time_budget_min:.0f} min reached after epoch {epoch + 1}; stopping")
            break
    log(f"done in {(time.time() - t0) / 60:.1f} min; backbone.pth written")


if __name__ == "__main__":
    main()


Overwriting src/pretrain.py


In [9]:
import py_compile
for f in ('src/train.py', 'src/pretrain.py'):
    py_compile.compile(f, doraise=True)
print('src/train.py, src/pretrain.py syntax OK')

src/train.py, src/pretrain.py syntax OK


## 4. Experiments and the job queue

Four stages with dependencies: **stage 1** (ImageNet-initialised runs, incl. the paired teachers) → **pretraining** on the archive → **stage 2** (SSL-initialised runs) → **stage 3** (distillation and hallucination students from the best paired teacher) → **CV** (five folds × three recipes). All runs: EfficientNet-V2-S @448, device heads, EMA, basic + clinical metadata.

In [2]:
if SDK_V3:
    from sagemaker.train import ModelTrainer
    from sagemaker.train.configs import SourceCode, Compute, InputData, StoppingCondition, OutputDataConfig
    from sagemaker.core import image_uris
else:
    from sagemaker.pytorch import PyTorch

ledger = json.load(open(LEDGER)) if os.path.exists(LEDGER) else {}
save_ledger = lambda: json.dump(ledger, open(LEDGER, 'w'), indent=2)
ACTIVE = {'InProgress', 'Stopping'}

def describe(tag):
    return sm.describe_training_job(TrainingJobName=ledger[tag]['job_name'])

def artifact_uri(tag):
    d = describe(tag)
    if d['TrainingJobStatus'] != 'Completed': raise RuntimeError(f'{tag} is {d["TrainingJobStatus"]}, cannot be used as an input')
    return d['ModelArtifacts']['S3ModelArtifacts']

def resolve_channels(ch):
    return {k: (v if str(v).startswith('s3://') else artifact_uri(v)) for k, v in (ch or {}).items()}

def launch(tag, hp, data_prefix, instance, entry='train.py', channels=None, max_hours=10):
    s3_in, s3_out = f's3://{BUCKET}/{data_prefix}/', f's3://{BUCKET}/{EXPERIMENT_ROOT}/{tag}/'
    chans = {'training': s3_in, **resolve_channels(channels)}
    base_job_name, max_run = f'ker7-{tag}'[:38], int(max_hours * 3600)
    if SDK_V3:
        image = image_uris.retrieve(framework='pytorch', region=REGION, version=FRAMEWORK_VERSION, py_version=PY_VERSION, instance_type=instance, image_scope='training')
        h = ModelTrainer(training_image=image, role=ROLE, base_job_name=base_job_name, sagemaker_session=SM_SESSION,
                         source_code=SourceCode(source_dir='src', entry_script=entry, requirements='requirements.txt'),
                         compute=Compute(instance_type=instance, instance_count=1, volume_size_in_gb=150),
                         stopping_condition=StoppingCondition(max_runtime_in_seconds=max_run),
                         output_data_config=OutputDataConfig(s3_output_path=s3_out), hyperparameters=hp)
        h.train(input_data_config=[InputData(channel_name=k, data_source=v) for k, v in chans.items()], wait=False, logs=False)
        job_name = h._latest_training_job.training_job_name
    else:
        h = PyTorch(entry_point=entry, source_dir='src', role=ROLE, instance_type=instance, instance_count=1, framework_version=FRAMEWORK_VERSION,
                    py_version=PY_VERSION, volume_size=150, max_run=max_run, base_job_name=base_job_name, hyperparameters=hp, output_path=s3_out,
                    sagemaker_session=SM_SESSION)
        h.fit(chans, wait=False, logs=False)
        job_name = h.latest_training_job.job_name
    ledger[tag] = dict(job_name=job_name, hp=hp, data_prefix=data_prefix, instance=instance, entry=entry, channels=chans,
                       launched=datetime.utcnow().isoformat(timespec='seconds'), output=s3_out)
    save_ledger()
    print(f'{datetime.now():%H:%M}  launched {tag:18s} on {instance:16s} -> {job_name}', flush=True)

BASE = {'img-size': 448, 'use-meta': 1, 'meta-set': 'all', 'meta-drop': 0.1, 'epochs': 20, 'batch-size': 16, 'grad-accum': 2,
        'head-lr': 3e-4, 'backbone-lr-mult': 0.1, 'weight-decay': 1e-2, 'dropout': 0.5, 'label-smoothing': 0.05,
        'patience': 6, 'warmup-epochs': 1.0, 'select-on': 'visit', 'ema': 1, 'ema-decay': 0.999, 'tta': 1,
        'workers': 8, 'seed': 42, 'time-budget-min': 90, 'labels-file': V7_LABELS, 'device-heads': 1, 'arch': 'efficientnet_v2_s'}
MULTI = {'cw-power': 0.5, 'patience': 7}
AUX   = {'aux-weight': 0.3}
PAIR  = {'views': 'pair', 'export-outputs': 1, 'batch-size': 12, 'grad-accum': 3, 'time-budget-min': 120}
KD    = {'kd-weight': 1.0, 'kd-temp': 2.0}
HAL   = {'halluc-weight': 1.0, 'halluc-init': 'teacher', 'batch-size': 12, 'grad-accum': 3, 'time-budget-min': 120}
PRETRAIN_HP = {'arch': 'efficientnet_v2_s', 'img-size': 224, 'epochs': 40, 'batch-size': 128, 'lr': 1e-3, 'temp': 0.1, 'pair-prob': 0.5,
               'workers': 14, 'time-budget-min': 200, 'manifest': ARCHIVE_MANIFEST}

E = lambda task, hp, channels=None: dict(hp={**hp, 'task': task}, data=DATA_PREFIX, channels=channels or {})
EXPERIMENTS = {
    # stage 1: ImageNet init
    'bf-base':       E('bf',    BASE),                                     # diffuse only; v6 bf-clin recipe under the new metric
    'bf-aux':        E('bf',    {**BASE, **AUX}),                          # + morphology auxiliary heads
    'bf-pair':       E('bf',    {**BASE, **PAIR}),                         # diffuse + blue (teacher candidate)
    'bf-pair-aux':   E('bf',    {**BASE, **PAIR, **AUX}),                  # diffuse + blue + aux (teacher candidate)
    'bfv-base':      E('bfv',   {**BASE, **MULTI}),
    'bfv-pair':      E('bfv',   {**BASE, **PAIR, **MULTI}),
    'tier1-pair':    E('tier1', {**BASE, **PAIR, **MULTI}),
    # stage 2: self-supervised init (channel 'pretrained' = the pretraining job's artifact)
    'bf-ssl':        E('bf',    BASE, {'pretrained': 'pretrain'}),
    'bf-ssl-aux':    E('bf',    {**BASE, **AUX}, {'pretrained': 'pretrain'}),
    'bf-ssl-pair-aux': E('bf',  {**BASE, **PAIR, **AUX}, {'pretrained': 'pretrain'}),
    'bfv-ssl-aux':   E('bfv',   {**BASE, **AUX, **MULTI}, {'pretrained': 'pretrain'}),
}
STAGE1 = ['bf-pair', 'bf-pair-aux', 'bfv-pair', 'tier1-pair', 'bf-base', 'bf-aux', 'bfv-base']
STAGE2 = ['bf-ssl-pair-aux', 'bf-ssl', 'bf-ssl-aux', 'bfv-ssl-aux']
HEAVY = {t for t, e in EXPERIMENTS.items() if 'views' in e['hp'] or 'halluc-weight' in e['hp']}       # two images or two backbones per step
G5_ONLY = set(HEAVY) | {'pretrain'}
ADAPT = {'ml.g5.4xlarge': lambda hp: {**hp, 'workers': 14},
         'ml.g5.2xlarge': lambda hp: hp,
         'ml.g4dn.2xlarge': lambda hp: {**hp, 'batch-size': max(1, hp['batch-size'] // 2), 'grad-accum': hp['grad-accum'] * 2,
                                        'time-budget-min': round(hp.get('time-budget-min', 0) * 1.6)}}

def running_per_instance():
    counts = {k: 0 for k in LIMITS}
    for tag, r in ledger.items():
        if r['instance'] in counts and describe(tag)['TrainingJobStatus'] in ACTIVE:
            counts[r['instance']] += 1
    return counts

def run_queue(order, poll_sec=120):
    pending = [t for t in order if t not in ledger]
    print(f'{len(order) - len(pending)} already launched, {len(pending)} queued: {pending}')
    failed_to_launch, last_state, polls = {}, None, 0
    while pending:
        counts = running_per_instance()
        free = [i for i in LIMITS if counts[i] < LIMITS[i]]
        launched = False
        for inst in free:
            heavy_ok = inst == 'ml.g5.4xlarge' or (inst == 'ml.g5.2xlarge')
            tag = next((t for t in pending if (t not in G5_ONLY) or heavy_ok), None)
            if tag is None: continue
            e = EXPERIMENTS[tag]
            try:
                launch(tag, ADAPT[inst](e['hp']), e['data'], inst, entry=e.get('entry', 'train.py'), channels=e.get('channels'))
                pending.remove(tag); launched = True; time.sleep(30)
            except Exception as err:
                if 'ResourceLimitExceeded' in str(err):
                    print(f'{datetime.now():%H:%M}  {inst} busy outside this ledger (or quota not yet active); waiting'); break
                failed_to_launch[tag] = str(err)[:300]; pending.remove(tag)
                print(f'{datetime.now():%H:%M}  could not launch {tag}: {str(err)[:200]}')
        if not launched:
            state = (tuple(sorted(counts.items())), len(pending))
            if state != last_state or polls % 15 == 0:
                print(f'{datetime.now():%H:%M}  running {counts} | queued {len(pending)}: {pending[:4]}{"..." if len(pending) > 4 else ""}', flush=True)
            last_state = state; polls += 1; time.sleep(poll_sec)
    print('all launched' + (f'; failed to launch: {failed_to_launch}' if failed_to_launch else ''))
    return failed_to_launch

def wait_for(tags, poll_sec=180, max_hours=18):
    tags = [t for t in tags if t in ledger]
    t0, last, polls = time.time(), None, 0
    while True:
        st = {t: describe(t)['TrainingJobStatus'] for t in tags}
        active = [t for t, s in st.items() if s in ACTIVE]
        if not active or time.time() - t0 > max_hours * 3600: break
        if (active, ) != last or polls % 10 == 0:
            print(f'{datetime.now():%H:%M}  waiting for {active}', flush=True)
        last = (active, ); polls += 1; time.sleep(poll_sec)
    bad = {t: s for t, s in st.items() if s not in ('Completed',)}
    print(f'{datetime.now():%H:%M}  finished: ' + ', '.join(f'{t}={s}' for t, s in st.items()))
    return bad

def job_status():
    rows = []
    for tag, r in ledger.items():
        d = describe(tag)
        rows.append(dict(tag=tag, status=d['TrainingJobStatus'], secondary=d.get('SecondaryStatus', ''), minutes=round(d.get('TrainingTimeInSeconds', 0) / 60),
                         instance=r['instance'], launched=r['launched'][11:16], failure=(d.get('FailureReason') or '')[:160]))
    return pd.DataFrame(rows)

def show_logs(tag, last=60):
    logs, group, job = boto3.client('logs'), '/aws/sagemaker/TrainingJobs', ledger[tag]['job_name']
    for st in logs.describe_log_streams(logGroupName=group, logStreamNamePrefix=job).get('logStreams', []):
        for e in logs.get_log_events(logGroupName=group, logStreamName=st['logStreamName'], limit=last, startFromHead=False)['events']:
            print(e['message'])

def best_by_val(tags, key='val_visit_auroc'):
    """Among completed runs, the one with the highest validation eye-visit AUROC (reads performance_summary.json from S3 artifacts)."""
    best, best_v = None, -1
    for t in tags:
        if t not in ledger or describe(t)['TrainingJobStatus'] != 'Completed': continue
        dest = f'{RUNS_DIR}/{t}'; os.makedirs(dest, exist_ok=True)
        if not os.path.exists(f'{dest}/performance_summary.json'):
            bucket, key_ = artifact_uri(t).replace('s3://', '').split('/', 1); s3.download_file(bucket, key_, f'{dest}/model.tar.gz')
            with tarfile.open(f'{dest}/model.tar.gz') as tf_: tf_.extractall(dest, members=[mm for mm in tf_.getmembers() if mm.name.endswith('performance_summary.json')])
            os.remove(f'{dest}/model.tar.gz')
        v = json.load(open(f'{dest}/performance_summary.json'))[key]
        if v > best_v: best, best_v = t, v
    return best, best_v

print(f'{len(EXPERIMENTS)} experiments defined; {len(ledger)} already in the ledger')

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
11 experiments defined; 41 already in the ledger


In [3]:
# --- recovery after a kernel restart: rebuild run definitions from the ledger ---
LIGHT = {'batch-size': 16, 'grad-accum': 2, 'time-budget-min': 90}
HEAVY_HP = {'batch-size': 12, 'grad-accum': 3, 'time-budget-min': 120}
for t, r in ledger.items():
    if t in EXPERIMENTS or t == 'pretrain' or t.startswith('cv-'): continue
    hp = dict(r['hp']); heavy = hp.get('views') == 'pair' or float(hp.get('halluc-weight', 0)) > 0
    hp.update(HEAVY_HP if heavy else LIGHT); hp['workers'] = 8
    EXPERIMENTS[t] = dict(hp=hp, data=r['data_prefix'], channels={k: v for k, v in r.get('channels', {}).items() if k != 'training'})
    if heavy: G5_ONLY.add(t)
print('recovered:', [t for t in EXPERIMENTS if t not in STAGE1 + STAGE2])
job_status()

recovered: ['bf-halluc', 'bf-kd-halluc', 'bf-kd', 'bfv-kd', 'cvt-f0', 'cvt-f1', 'cvt-f2', 'cvt-f3', 'cvt-f4', 'cvs-f0', 'cvs-f1', 'cvs-f2', 'cvs-f3', 'cvs-f4']


,tag,status,secondary,minutes,instance,launched,failure
0,bf-pair,Completed,Completed,30,ml.g5.4xlarge,03:58,
1,bf-pair-aux,Completed,Completed,38,ml.g5.2xlarge,03:58,
2,bf-base,Completed,Completed,29,ml.g4dn.2xlarge,03:59,
3,bfv-pair,Completed,Completed,43,ml.g5.4xlarge,03:59,
4,tier1-pair,Completed,Completed,42,ml.g5.2xlarge,04:00,
5,bf-aux,Completed,Completed,31,ml.g4dn.2xlarge,04:00,
6,bfv-base,Completed,Completed,21,ml.g5.2xlarge,04:01,
7,pretrain,Completed,Completed,205,ml.g5.4xlarge,06:08,
8,bf-ssl-pair-aux,Completed,Completed,30,ml.g5.4xlarge,11:50,
9,bf-ssl,Completed,Completed,16,ml.g5.2xlarge,11:50,


**4a. Stage 1** — launches and returns (the jobs run while the archive is staged below).

In [11]:
failed_stage1 = run_queue(STAGE1)

0 already launched, 7 queued: ['bf-pair', 'bf-pair-aux', 'bfv-pair', 'tier1-pair', 'bf-base', 'bf-aux', 'bfv-base']
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
03:58  launched bf-pair            on ml.g5.4xlarge    -> ker7-bf-pair-20261014035811
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
03:58  launched bf-pair-aux        on ml.g5.2xlarge    -> ker7-bf-pair-aux-20261014035845
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
03:59  launched bf-base            on ml.g4dn.2xlarge  -> ker7-bf-base-20261014035918
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
03:59  launched bfv-pair           on ml.g5.4xlarge 

**4b. Archive staging and self-supervised pretraining.** Every image in the bucket root (all illuminations, labelled or not, excluding photographed forms) is staged at 384 px into `v7-archive-384/` with a manifest that records the diffuse ↔ blue partner of the same eye-visit where one exists. Resumable; ~2–3 h for ~60k originals at 3 workers. Then the pretraining job is launched on a g5.4xlarge.

In [12]:
def build_archive():
    if not FORCE_REBUILD and s3_exists(f'{ARCHIVE_PREFIX}/{ARCHIVE_MANIFEST}'):
        s3.download_file(BUCKET, f'{ARCHIVE_PREFIX}/{ARCHIVE_MANIFEST}', ARCHIVE_MANIFEST); am = pd.read_csv(ARCHIVE_MANIFEST)
        print(f'{ARCHIVE_PREFIX}: manifest already in S3 ({len(am):,} images) -- reusing it'); return am
    root = list_root_images()
    r = raw[~raw['non_eye']].copy(); r['image_name'] = r['id'].map(root); r = r[r['image_name'].notna()]
    r['visit_id'] = r['visit_id'].astype(str); r['kind'] = np.where(r['is_blue'], 'blue', np.where(r['illum'].eq('diffuse'), 'diffuse', 'other'))
    # partner of the other illumination within the same (patient, visit, eye); same device preferred, nearest in time
    a = r[r.kind == 'diffuse'][['image_name', 'patient_id', 'visit_id', 'eye_l', 'device_n', 'img_date']]
    b = r[r.kind == 'blue'][['image_name', 'patient_id', 'visit_id', 'eye_l', 'device_n', 'img_date']].rename(columns={'image_name': 'pair', 'device_n': 'dev2', 'img_date': 'date2'})
    c = a.merge(b, on=['patient_id', 'visit_id', 'eye_l'])
    c['same'] = (c['device_n'] == c['dev2']).astype(int); c['dt'] = (c['img_date'] - c['date2']).abs().dt.total_seconds().fillna(1e12)
    c = c.sort_values(['image_name', 'same', 'dt'], ascending=[True, False, True]).drop_duplicates('image_name')
    pairs = dict(zip(c['image_name'], c['pair'])); pairs.update({v: k for k, v in pairs.items()})          # both directions
    other = set(r.loc[r.kind == 'other', 'image_name'])
    names = [k for k in root.values() if k not in other]                                               # keep diffuse, blue and unlisted images
    if MAX_ARCHIVE and len(names) > MAX_ARCHIVE:
        rng = np.random.default_rng(SPLIT_SEED); keep = set(rng.choice(names, MAX_ARCHIVE, replace=False)); keep |= {pairs[n] for n in keep if n in pairs}
        names = [n for n in names if n in keep]
    print(f'archive: {len(names):,} images ({len(pairs) // 2:,} diffuse-blue pairs)')
    staged, failed = stage_images(names, ARCHIVE_PREFIX, ARCHIVE_SHORT_SIDE)
    am = pd.DataFrame({'image_name': [n for n in names if n in staged]})
    am['pair'] = am['image_name'].map(pairs).fillna(''); am.loc[~am['pair'].isin(staged), 'pair'] = ''
    am.to_csv(ARCHIVE_MANIFEST, index=False); s3.upload_file(ARCHIVE_MANIFEST, BUCKET, f'{ARCHIVE_PREFIX}/{ARCHIVE_MANIFEST}')
    print(f'  archive manifest: {len(am):,} images, {int((am.pair != "").sum()):,} with a partner ({len(failed)} failed)')
    return am

ARCHIVE = build_archive()
EXPERIMENTS['pretrain'] = dict(hp=PRETRAIN_HP, data=ARCHIVE_PREFIX, channels={}, entry='pretrain.py')
if 'pretrain' not in ledger:
    launch('pretrain', ADAPT['ml.g5.4xlarge'](PRETRAIN_HP), ARCHIVE_PREFIX, 'ml.g5.4xlarge', entry='pretrain.py')

archive: 67,569 images (29,082 diffuse-blue pairs)
  staging -> v7-archive-384/ @ 384px: 0 present, 67,569 to process
    1,000/67,569  failed 75  1.7 min  eta 112 min
    2,000/67,569  failed 163  3.4 min  eta 112 min
    3,000/67,569  failed 246  5.1 min  eta 110 min
    4,000/67,569  failed 306  6.8 min  eta 108 min
    5,000/67,569  failed 368  8.5 min  eta 106 min
    6,000/67,569  failed 445  10.2 min  eta 105 min
    7,000/67,569  failed 538  11.9 min  eta 103 min
    8,000/67,569  failed 601  13.6 min  eta 101 min
    9,000/67,569  failed 601  15.7 min  eta 102 min
    10,000/67,569  failed 601  17.8 min  eta 102 min
    11,000/67,569  failed 601  19.8 min  eta 102 min
    12,000/67,569  failed 601  21.9 min  eta 102 min
    13,000/67,569  failed 601  24.1 min  eta 101 min
    14,000/67,569  failed 601  26.2 min  eta 100 min
    15,000/67,569  failed 601  28.3 min  eta 99 min
    16,000/67,569  failed 601  30.5 min  eta 98 min
    17,000/67,569  failed 601  32.6 min  eta 97 min

**4c. Stage 2 (SSL runs) after pretraining; stage 3 (students) after the best paired teacher.** The teacher for the students is the paired run with the highest validation eye-visit AUROC (ImageNet- or SSL-initialised).

In [ ]:
bad = wait_for(['pretrain'])
if not bad:
    run_queue(STAGE2)
bad = wait_for(STAGE1 + STAGE2)
TEACHER_BF, v = best_by_val(['bf-pair', 'bf-pair-aux', 'bf-ssl-pair-aux'])
TEACHER_BFV, v3 = best_by_val(['bfv-pair'])
print(f'teacher for bf students: {TEACHER_BF} (val visit AUROC {v:.3f}); for bfv: {TEACHER_BFV}')
BEST_DIFFUSE, vd = best_by_val(['bf-base', 'bf-aux', 'bf-ssl', 'bf-ssl-aux'])
BEST_DIFFUSE = BEST_DIFFUSE or 'bf-base'
print(f'best diffuse-only recipe so far: {BEST_DIFFUSE} (val visit AUROC {vd:.3f})')
SSL_CH = {'pretrained': 'pretrain'} if 'pretrain' in ledger and describe('pretrain')['TrainingJobStatus'] == 'Completed' else {}
best_hp = EXPERIMENTS[BEST_DIFFUSE]['hp']; best_ch = EXPERIMENTS[BEST_DIFFUSE]['channels']
EXPERIMENTS.update({
    'bf-kd':         E('bf', {**best_hp, **KD}, {**best_ch, 'teacher': TEACHER_BF}),                # best diffuse recipe + distillation
    'bf-halluc':     E('bf', {**best_hp, **HAL}, {**best_ch, 'teacher': TEACHER_BF}),               # best diffuse recipe + hallucination branch
    'bf-kd-halluc':  E('bf', {**best_hp, **KD, **HAL}, {**best_ch, 'teacher': TEACHER_BF}),
    'bfv-kd':        E('bfv', {**BASE, **AUX, **MULTI, **KD}, {**SSL_CH, 'teacher': TEACHER_BFV}),
})
G5_ONLY |= {'bf-halluc', 'bf-kd-halluc'}
STAGE3 = ['bf-halluc', 'bf-kd-halluc', 'bf-kd', 'bfv-kd']
run_queue(STAGE3)
bad = wait_for(STAGE3)
job_status()

06:08  waiting for ['pretrain']
06:38  waiting for ['pretrain']
07:08  waiting for ['pretrain']
07:38  waiting for ['pretrain']
08:08  waiting for ['pretrain']
08:38  waiting for ['pretrain']
09:08  waiting for ['pretrain']
09:38  waiting for ['pretrain']
10:08  waiting for ['pretrain']
10:38  waiting for ['pretrain']
11:08  waiting for ['pretrain']
11:38  waiting for ['pretrain']
11:50  finished: pretrain=Completed
0 already launched, 4 queued: ['bf-ssl-pair-aux', 'bf-ssl', 'bf-ssl-aux', 'bfv-ssl-aux']
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
11:50  launched bf-ssl-pair-aux    on ml.g5.4xlarge    -> ker7-bf-ssl-pair-aux-20261014115013
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
11:50  launched bf-ssl             on ml.g5.2xlarge    -> ker7-bf-ssl-20261014115046
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
11:51  launched bf-ssl-aux         on ml.g

**4d. Cross-validation stage.** Five folds over *all* patients (train + val + test pooled; fold k held out as test, a grouped 15 % slice of the rest as validation) for three recipes: the diffuse-only reference, the best diffuse-only recipe (students included), and the best paired recipe. Out-of-fold predictions are pooled in Section 5 and compared with a paired bootstrap.

In [5]:
RUN_CV = True
CV_RECIPES = {}
if RUN_CV:
    BEST_DEPLOY, vdep = best_by_val([t for t in ['bf-base', 'bf-aux', 'bf-ssl', 'bf-ssl-aux', 'bf-kd', 'bf-halluc', 'bf-kd-halluc'] if t in ledger])
    BEST_PAIR, vpair = best_by_val([t for t in ['bf-pair', 'bf-pair-aux', 'bf-ssl-pair-aux'] if t in ledger])
    CV_RECIPES = {'cv-base': 'bf-base', 'cv-best': BEST_DEPLOY, 'cv-pair': BEST_PAIR}
    print(f'CV recipes: {CV_RECIPES}')
    cv_tags = []
    for cv_name, src in CV_RECIPES.items():
        if src is None or src == 'bf-base' and cv_name != 'cv-base': continue
        for k in range(N_FOLDS):
            hp = {**EXPERIMENTS[src]['hp'], 'fold': k, 'n-folds': N_FOLDS, 'cv-pool': 'all', 'export-outputs': 0}
            EXPERIMENTS[f'{cv_name}-f{k}'] = E(hp['task'], hp, EXPERIMENTS[src]['channels'])
            if src in G5_ONLY: G5_ONLY.add(f'{cv_name}-f{k}')
            cv_tags.append(f'{cv_name}-f{k}')
    run_queue(cv_tags)
    bad = wait_for(cv_tags)
job_status()

CV recipes: {'cv-base': 'bf-base', 'cv-best': 'bf-kd-halluc', 'cv-pair': 'bf-ssl-pair-aux'}
15 already launched, 0 queued: []
all launched
01:05  finished: cv-base-f0=Completed, cv-base-f1=Completed, cv-base-f2=Completed, cv-base-f3=Completed, cv-base-f4=Completed, cv-best-f0=Completed, cv-best-f1=Completed, cv-best-f2=Completed, cv-best-f3=Completed, cv-best-f4=Completed, cv-pair-f0=Completed, cv-pair-f1=Completed, cv-pair-f2=Completed, cv-pair-f3=Completed, cv-pair-f4=Completed


,tag,status,secondary,minutes,instance,launched,failure
0,bf-pair,Completed,Completed,30,ml.g5.4xlarge,03:58,
1,bf-pair-aux,Completed,Completed,38,ml.g5.2xlarge,03:58,
2,bf-base,Completed,Completed,29,ml.g4dn.2xlarge,03:59,
3,bfv-pair,Completed,Completed,43,ml.g5.4xlarge,03:59,
4,tier1-pair,Completed,Completed,42,ml.g5.2xlarge,04:00,
5,bf-aux,Completed,Completed,31,ml.g4dn.2xlarge,04:00,
6,bfv-base,Completed,Completed,21,ml.g5.2xlarge,04:01,
7,pretrain,Completed,Completed,205,ml.g5.4xlarge,06:08,
8,bf-ssl-pair-aux,Completed,Completed,30,ml.g5.4xlarge,11:50,
9,bf-ssl,Completed,Completed,16,ml.g5.2xlarge,11:50,


## 5. Results

Reported: index-visit (first labelled visit), image level, and the patient-level cross-visit average for continuity. Paired-subset columns score models on the test eye-visits that have a blue-light pair, so the teacher and its students are compared on identical rows.

In [8]:
# ===== Section 5 common cell =====
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression
 
if 'ledger' not in globals():
    ledger = json.load(open(LEDGER)); describe = lambda tag: sm.describe_training_job(TrainingJobName=ledger[tag]['job_name'])
if 'MAN7' not in globals():
    s3.download_file(BUCKET, f'{DATA_PREFIX}/{V7_ALL}', V7_ALL); MAN7 = pd.read_csv(V7_ALL); MAN7['patient_id'] = pid_str(MAN7['patient_id'])
    USE7 = MAN7[MAN7['eye_status'].isin(CLEAN_STATUS)]
CLASSES = {'bf': ['Bacterial', 'Fungal'], 'bfv': ['Bacterial', 'Fungal', 'Viral'], 'tier1': ['NonViral', 'Viral']}
YMAP = {'bf': {'Bacterial': 0, 'Fungal': 1}, 'bfv': {'Bacterial': 0, 'Fungal': 1, 'Viral': 2}, 'tier1': {'Bacterial': 0, 'Fungal': 0, 'Viral': 1}}
VKEYS = ['patient_id', 'visit_id', 'eye']
 
def fetch(tag, force=False):
    dest, summ = f'{RUNS_DIR}/{tag}', f'{RUNS_DIR}/{tag}/performance_summary.json'
    if force or not os.path.exists(f'{dest}/test_predictions.csv'):
        d = describe(tag)
        if d['TrainingJobStatus'] != 'Completed': return None
        os.makedirs(dest, exist_ok=True)
        bucket, key = d['ModelArtifacts']['S3ModelArtifacts'].replace('s3://', '').split('/', 1)
        tar_path = f'{dest}/model.tar.gz'; s3.download_file(bucket, key, tar_path)
        with tarfile.open(tar_path) as t:
            t.extractall(dest, members=[mm for mm in t.getmembers() if not mm.name.endswith(('.pth', '.npz'))])
        os.remove(tar_path); print('fetched', tag, flush=True)
    return json.load(open(summ)) if os.path.exists(summ) else None
 
def preds(tag, split='test', runs_dir=None):
    d = pd.read_csv(f'{runs_dir or RUNS_DIR}/{tag}/{split}_predictions.csv'); d['patient_id'] = pid_str(d['patient_id'])
    for c in ('is_index', 'has_blue'):
        if c not in d.columns: d = d.merge(MAN7[['image_name', c]], on='image_name', how='left')
    d['eye'] = d['eye'].astype(str).str.lower(); return d
 
P_of = lambda d, classes: d[[f'prob_{c}' for c in classes]].values
 
def auroc(y, P):
    y, P = np.asarray(y), np.asarray(P)
    if P.shape[1] == 2: return roc_auc_score(y, P[:, 1]) if len(np.unique(y)) > 1 else np.nan
    vals = [roc_auc_score(y == c, P[:, c]) for c in range(P.shape[1]) if 0 < (y == c).sum() < len(y)]
    return float(np.mean(vals)) if vals else np.nan
 
def per_class(y, P):
    return [roc_auc_score(y == c, P[:, c]) if 0 < (y == c).sum() < len(y) else np.nan for c in range(P.shape[1])]
 
def boot_idx(groups, n_boot, seed):
    rng = np.random.default_rng(seed); groups = np.asarray(groups); uniq = np.unique(groups); idx_by = {g: np.where(groups == g)[0] for g in uniq}
    for _ in range(n_boot):
        yield np.concatenate([idx_by[g] for g in rng.choice(uniq, size=len(uniq), replace=True)])
 
def boot_ci(y, P, groups, n_boot=1000, seed=0):
    y, P = np.asarray(y), np.asarray(P); vals = [auroc(y[i], P[i]) for i in boot_idx(groups, n_boot, seed)]; vals = [v for v in vals if not np.isnan(v)]
    return tuple(np.percentile(vals, [2.5, 97.5])) if len(vals) >= 50 else (np.nan, np.nan)
 
def paired_diff(y, P1, P2, groups, n_boot=2000, seed=0):
    """AUROC(P2) - AUROC(P1) with a patient-cluster bootstrap CI and the share of resamples where P2 wins."""
    y, P1, P2 = np.asarray(y), np.asarray(P1), np.asarray(P2)
    d = [auroc(y[i], P2[i]) - auroc(y[i], P1[i]) for i in boot_idx(groups, n_boot, seed)]; d = np.array([v for v in d if not np.isnan(v)])
    return float(auroc(y, P2) - auroc(y, P1)), tuple(np.percentile(d, [2.5, 97.5])), float((d > 0).mean())
 
def rollup(d, classes, by):
    pc = [f'prob_{c}' for c in classes]
    agg = {**{c: (c, 'mean') for c in pc}, 'label': ('label', 'first'), 'n': ('label', 'size')}
    for c in ('site', 'device', 'eye_status', 'has_blue', 'is_index', 'patient_id'):
        if c in d.columns and c not in by: agg[c] = (c, lambda s: s.mode().iloc[0])
    return d.groupby(by).agg(**agg).reset_index()
 
def levels(d, classes):
    v = rollup(d, classes, VKEYS)
    return {'image': d, 'visit': v, 'index_visit': v[pd.to_numeric(v['is_index'], errors='coerce').fillna(0).astype(int) == 1], 'patient': rollup(d, classes, ['patient_id'])}
 
def score(d, classes, ymap, with_ci=True):
    d = d[d['label'].isin(ymap)].copy(); d['y'] = d['label'].map(ymap); out = {}
    for level, f in levels(d, classes).items():
        if level == 'image': continue                                  # image level not reported
        f = f.assign(y=f['label'].map(ymap)); P, y = P_of(f, classes), f['y'].values
        res = {'n': len(f), 'auroc': auroc(y, P)}
        if with_ci: res['ci'] = boot_ci(y, P, f['patient_id'].values)
        if len(classes) > 2: res['per_class'] = dict(zip(classes, per_class(y, P)))
        out[level] = res
    v = levels(d, classes)['visit'].assign(y=lambda f: f['label'].map(ymap))
    for col in ('device', 'has_blue', 'eye_status'):                   # stratified visit-level AUROC
        if col in v.columns:
            for val, g in v.groupby(v[col].astype(str)):
                out[f'{col}={val}'] = round(auroc(g['y'], P_of(g, classes)), 3)
    return out
 
fmt = lambda r: f"{r['auroc']:.3f} [{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]" if 'ci' in r and not np.isnan(r['ci'][0]) else f"{r['auroc']:.3f}"
 
SUMMARIES = {t: fetch(t) for t in ledger if t != 'pretrain'}
DONE = {t: s for t, s in SUMMARIES.items() if s}
missing = [t for t in ledger if t not in DONE and t != 'pretrain']
print(f'completed runs: {sorted(DONE)}' + (f'\nnot completed (skipped): {missing}' if missing else ''))
task_runs = lambda task, cv=False: [t for t, s in DONE.items() if s['task'] == task and (t.startswith(('cv-', 'cvs-', 'cvt-')) == cv)] 
def ensemble_P(tags, classes, split='test'):
    frames = [preds(t, split) for t in tags]; base = frames[0].drop(columns=[c for c in frames[0].columns if c.startswith('prob_')]).copy()
    Ps = []
    for f in frames:
        f = f.set_index('image_name').reindex(base['image_name']); assert not f['label'].isna().any(), 'ensemble members cover different rows'; Ps.append(P_of(f, classes))
    P = np.mean(Ps, axis=0)
    for j, c in enumerate(classes): base[f'prob_{c}'] = P[:, j]
    return base
 
def v6_preds(tag, split='test'):
    f = f'{V6_RUNS_DIR}/{tag}/{split}_predictions.csv'
    if not os.path.exists(f): return None
    d = pd.read_csv(f); d['patient_id'] = pid_str(d['patient_id']); keep = set(USE7.loc[USE7.split == split, 'image_name'])
    d = d[d['image_name'].isin(keep)].drop(columns=[c for c in ('is_index', 'has_blue', 'eye_status') if c in d.columns])
    d = d.merge(MAN7[['image_name', 'is_index', 'has_blue', 'eye_status']], on='image_name', how='left'); d['eye'] = d['eye'].astype(str).str.lower()
    return d

completed runs: ['bf-aux', 'bf-base', 'bf-halluc', 'bf-kd', 'bf-kd-halluc', 'bf-pair', 'bf-pair-aux', 'bf-ssl', 'bf-ssl-aux', 'bf-ssl-pair-aux', 'bfv-base', 'bfv-kd', 'bfv-pair', 'bfv-ssl-aux', 'cv-base-f0', 'cv-base-f1', 'cv-base-f2', 'cv-base-f3', 'cv-base-f4', 'cv-best-f0', 'cv-best-f1', 'cv-best-f2', 'cv-best-f3', 'cv-best-f4', 'cv-pair-f0', 'cv-pair-f1', 'cv-pair-f2', 'cv-pair-f3', 'cv-pair-f4', 'cvs-f0', 'cvs-f1', 'cvs-f2', 'cvs-f3', 'cvs-f4', 'cvt-f0', 'cvt-f1', 'cvt-f2', 'cvt-f3', 'cvt-f4', 'tier1-pair']


### 5a. Bacterial vs fungal

In [9]:
# ===== 5a Bacterial vs fungal =====
rows = []
def bf_row(name, d, extra='', tag=None):
    r = score(d, CLASSES['bf'], YMAP['bf'])
    row = dict(model=name, visit=fmt(r['visit']), index_visit=fmt(r['index_visit']), patient=fmt(r['patient']),
               n_visit=r['visit']['n'], n_index=r['index_visit']['n'], n_pat=r['patient']['n'])
    row.update({k: v for k, v in r.items() if '=' in k})
    if tag:
        s = DONE[tag]; row.update(val_visit=round(s['val_visit_auroc'], 3), best_ep=s['best_epoch'], n_train=s['n_train'], inst=ledger[tag]['instance'][3:], min=round(s['train_minutes']))
    row['notes'] = extra; return row
 
bf_tags = task_runs('bf')
for t in bf_tags:
    s = DONE[t]; rows.append(bf_row(t, preds(t), ('paired input' if s['views'] == 'pair' else 'diffuse only') + (' | aux' if s['aux_cols'] else '')
                             + (' | KD' if s['hyperparameters'].get('kd_weight', 0) > 0 else '') + (' | halluc' if s['halluc'] else '')
                             + (' | SSL init' if 'pretrained' in ledger[t].get('channels', {}) else ''), tag=t))
diffuse = [t for t in bf_tags if DONE[t]['views'] != 'pair']; paired = [t for t in bf_tags if DONE[t]['views'] == 'pair']
if len(diffuse) >= 2:
    rows.append(bf_row('ENSEMBLE diffuse-only runs', ensemble_P(diffuse, CLASSES['bf']), '+'.join(diffuse)))
    top2 = sorted(diffuse, key=lambda t: -DONE[t]['val_visit_auroc'])[:2]
    rows.append(bf_row('ENSEMBLE top-2 diffuse by val', ensemble_P(top2, CLASSES['bf']), '+'.join(top2)))
if len(paired) >= 2:
    rows.append(bf_row('ENSEMBLE paired runs', ensemble_P(paired, CLASSES['bf']), '+'.join(paired)))
if diffuse and paired:
    rows.append(bf_row('ENSEMBLE best diffuse + best paired', ensemble_P([max(diffuse, key=lambda t: DONE[t]['val_visit_auroc']), max(paired, key=lambda t: DONE[t]['val_visit_auroc'])], CLASSES['bf']), 'needs blue at inference'))
for t in ('bf-clin', 'bf-img'):
    d = v6_preds(t)
    if d is not None and len(d): rows.append(bf_row(f'v6 {t} rescored on v7 rows', d, 'v6 training, same test rows, visit-level metric'))
 
bf_table = pd.DataFrame(rows); bf_table.to_csv(f'{RUNS_DIR}/results_bf.csv', index=False)
print(f'Bacterial vs Fungal -- target {TARGETS["bf"]:.2f} (eye-visit AUROC)')
display(bf_table)
 
# paired-subset comparison: everything scored on the test eye-visits that HAVE a blue pair (identical rows for teacher and students)
sub = []
for t in bf_tags:
    d = preds(t); d = d[d['has_blue'] == 1]
    r = score(d, CLASSES['bf'], YMAP['bf']); sub.append(dict(model=t, views=DONE[t]['views'], visit_paired=fmt(r['visit']), index_paired=fmt(r['index_visit']), n_visit=r['visit']['n']))
paired_table = pd.DataFrame(sub).sort_values('visit_paired', ascending=False); paired_table.to_csv(f'{RUNS_DIR}/results_bf_paired_subset.csv', index=False)
print('\nTest eye-visits with a blue-light pair only:'); display(paired_table)
 
# what did the auxiliary heads learn? (test AUROC per morphology target, per run)
aux = {t: DONE[t]['aux_test_auroc'] for t in bf_tags if DONE[t].get('aux_test_auroc')}
if aux:
    aux_table = pd.DataFrame(aux).round(3); aux_table.to_csv(f'{RUNS_DIR}/results_aux_heads.csv')
    print('\nAuxiliary morphology heads -- test AUROC of each target (how learnable each sign is from the image):'); display(aux_table)

Bacterial vs Fungal -- target 0.88 (eye-visit AUROC)


,model,visit,index_visit,patient,n_visit,n_index,n_pat,device=slit_lamp,device=smartphone,has_blue=0,has_blue=1,eye_status=infected,eye_status=infected_propagated,eye_status=single_unmatched,val_visit,best_ep,n_train,inst,min,notes
0,bf-pair,"0.799 [0.752, 0.843]","0.829 [0.789, 0.868]","0.829 [0.784, 0.868]",674,389,388,0.808,0.777,0.762,0.801,0.808,0.742,0.667,0.820,5.0,5942.0,g5.4xlarge,23.0,paired input
1,bf-pair-aux,"0.811 [0.767, 0.853]","0.833 [0.791, 0.873]","0.838 [0.796, 0.880]",674,389,388,0.832,0.752,0.796,0.812,0.819,0.777,0.667,0.821,8.0,5942.0,g5.2xlarge,30.0,paired input | aux
2,bf-base,"0.799 [0.753, 0.843]","0.825 [0.781, 0.866]","0.823 [0.777, 0.867]",674,389,388,0.811,0.770,0.805,0.798,0.812,0.752,0.667,0.818,5.0,5942.0,g4dn.2xlarge,23.0,diffuse only
3,bf-aux,"0.801 [0.755, 0.845]","0.821 [0.775, 0.862]","0.817 [0.772, 0.860]",674,389,388,0.814,0.775,0.808,0.801,0.811,0.768,0.000,0.831,6.0,5942.0,g4dn.2xlarge,25.0,diffuse only | aux
4,bf-ssl-pair-aux,"0.818 [0.771, 0.864]","0.856 [0.821, 0.892]","0.845 [0.804, 0.883]",674,389,388,0.830,0.781,0.810,0.819,0.829,0.781,0.667,0.839,4.0,5942.0,g5.4xlarge,21.0,paired input | aux | SSL init
5,bf-ssl,"0.792 [0.745, 0.841]","0.833 [0.789, 0.872]","0.824 [0.781, 0.866]",674,389,388,0.811,0.734,0.785,0.792,0.807,0.746,0.667,0.835,3.0,5942.0,g5.2xlarge,10.0,diffuse only | SSL init
6,bf-ssl-aux,"0.808 [0.761, 0.858]","0.854 [0.813, 0.890]","0.842 [0.800, 0.881]",674,389,388,0.828,0.750,0.801,0.808,0.825,0.749,0.667,0.837,6.0,5942.0,g4dn.2xlarge,25.0,diffuse only | aux | SSL init
7,bf-halluc,"0.796 [0.750, 0.845]","0.841 [0.799, 0.878]","0.829 [0.784, 0.869]",674,389,388,0.820,0.722,0.781,0.798,0.813,0.748,0.667,0.837,2.0,5942.0,g5.4xlarge,17.0,diffuse only | aux | halluc | SSL init
8,bf-kd-halluc,"0.819 [0.772, 0.866]","0.862 [0.825, 0.897]","0.849 [0.807, 0.886]",674,389,388,0.837,0.766,0.790,0.820,0.834,0.766,0.667,0.840,9.0,5942.0,g5.2xlarge,32.0,diffuse only | aux | KD | halluc | SSL init
9,bf-kd,"0.802 [0.755, 0.852]","0.846 [0.805, 0.884]","0.834 [0.790, 0.873]",674,389,388,0.820,0.746,0.796,0.802,0.816,0.758,0.667,0.839,6.0,5942.0,g4dn.2xlarge,25.0,diffuse only | aux | KD | SSL init



Test eye-visits with a blue-light pair only:


,model,views,visit_paired,index_paired,n_visit
8,bf-kd-halluc,diffuse,"0.820 [0.770, 0.869]","0.868 [0.829, 0.903]",646
4,bf-ssl-pair-aux,pair,"0.818 [0.768, 0.867]","0.859 [0.817, 0.897]",646
1,bf-pair-aux,pair,"0.809 [0.763, 0.852]","0.834 [0.794, 0.873]",646
6,bf-ssl-aux,diffuse,"0.808 [0.756, 0.861]","0.859 [0.819, 0.896]",646
9,bf-kd,diffuse,"0.802 [0.748, 0.855]","0.851 [0.812, 0.890]",646
3,bf-aux,diffuse,"0.801 [0.754, 0.848]","0.824 [0.779, 0.863]",646
7,bf-halluc,diffuse,"0.799 [0.751, 0.848]","0.846 [0.805, 0.884]",646
2,bf-base,diffuse,"0.798 [0.750, 0.846]","0.828 [0.786, 0.867]",646
0,bf-pair,pair,"0.797 [0.749, 0.842]","0.828 [0.786, 0.869]",646
5,bf-ssl,diffuse,"0.793 [0.741, 0.844]","0.837 [0.795, 0.878]",646



Auxiliary morphology heads -- test AUROC of each target (how learnable each sign is from the image):


,bf-pair-aux,bf-aux,bf-ssl-pair-aux,bf-ssl-aux,bf-halluc,bf-kd-halluc,bf-kd
depth_mid,0.748,0.747,0.780,0.781,0.776,0.788,0.783
depth_post,0.844,0.834,0.868,0.874,0.867,0.880,0.875
dry_surface,0.729,0.676,0.702,0.702,0.684,0.718,0.699
endothelial_plaque,0.750,0.722,0.826,0.831,0.759,0.851,0.828
epi_any,0.801,0.707,0.852,0.806,0.826,0.820,0.812
epi_ge2,0.843,0.812,0.870,0.855,0.845,0.864,0.856
epi_ge6,0.887,0.864,0.917,0.913,0.905,0.922,0.914
feathery,0.730,0.703,0.773,0.781,0.774,0.785,0.780
hypopyon,0.759,0.770,0.834,0.839,0.839,0.845,0.842
hypopyon_ge2mm,0.813,0.818,0.900,0.907,0.887,0.920,0.910


### 5b. Three-class and viral tier

In [10]:
# ===== 5b Three-class and viral tier =====
C3, Y3 = CLASSES['bfv'], YMAP['bfv']
def two_tier_P(t1, bf):
    t1 = t1.set_index('image_name'); bf = bf.set_index('image_name'); common = t1.index.intersection(bf.index)
    out = t1.loc[common].drop(columns=[c for c in t1.columns if c.startswith('prob_')]).reset_index()
    pv = t1.loc[common, 'prob_Viral'].values
    out['prob_Bacterial'] = (1 - pv) * bf.loc[common, 'prob_Bacterial'].values; out['prob_Fungal'] = (1 - pv) * bf.loc[common, 'prob_Fungal'].values; out['prob_Viral'] = pv
    return out
def row3(name, kind, d, notes=''):
    r = score(d, C3, Y3); pc = r['visit'].get('per_class', {})
    row = dict(system=name, kind=kind, visit_macro=fmt(r['visit']), index_macro=fmt(r['index_visit']), patient_macro=fmt(r['patient']),
               visit_B=round(pc.get('Bacterial', np.nan), 3), visit_F=round(pc.get('Fungal', np.nan), 3), visit_V=round(pc.get('Viral', np.nan), 3), n_visit=r['visit']['n'])
    row.update({k: v for k, v in r.items() if '=' in k}); row['notes'] = notes; return row
rows = []
bfv_tags = task_runs('bfv')
for t in bfv_tags:
    rows.append(row3(t, 'direct', preds(t), f"{DONE[t]['views']} | val visit {DONE[t]['val_visit_auroc']:.3f}"))
if len(bfv_tags) >= 2: rows.append(row3('ENSEMBLE direct', 'direct', ensemble_P(bfv_tags, C3), '+'.join(bfv_tags)))
for t1 in task_runs('tier1'):
    r = score(preds(t1), CLASSES['tier1'], YMAP['tier1'])
    rows.append(dict(system=t1, kind='tier 1 alone (viral vs rest)', visit_macro=fmt(r['visit']), index_macro=fmt(r['index_visit']), patient_macro=fmt(r['patient']), n_visit=r['visit']['n'], notes='binary AUROC'))
    for bf in bf_tags:
        if DONE[bf]['views'] == DONE[t1]['views']: rows.append(row3(f'{t1} -> {bf}', 'two-tier', two_tier_P(preds(t1), preds(bf))))
d = v6_preds('bfv-clin')
if d is not None and len(d): rows.append(row3('v6 bfv-clin rescored on v7 rows', 'direct', d, 'v6 training, same rows'))
bfv_table = pd.DataFrame(rows); bfv_table.to_csv(f'{RUNS_DIR}/results_bfv.csv', index=False)
print(f'Bacterial vs Fungal vs Viral -- target macro AUROC {TARGETS["bfv"]:.2f} (eye-visit level)'); display(bfv_table)


Bacterial vs Fungal vs Viral -- target macro AUROC 0.90 (eye-visit level)


,system,kind,visit_macro,index_macro,patient_macro,visit_B,visit_F,visit_V,n_visit,device=slit_lamp,device=smartphone,has_blue=0,has_blue=1,eye_status=infected,eye_status=infected_propagated,eye_status=single_unmatched,notes
0,bfv-pair,direct,"0.828 [0.794, 0.857]","0.856 [0.829, 0.884]","0.858 [0.830, 0.884]",0.741,0.860,0.884,785,0.846,0.782,0.801,0.830,0.830,0.813,0.500,pair | val visit 0.865
1,bfv-base,direct,"0.823 [0.787, 0.853]","0.847 [0.816, 0.875]","0.850 [0.818, 0.878]",0.738,0.845,0.886,785,0.839,0.780,0.746,0.827,0.829,0.795,0.500,diffuse | val visit 0.856
2,bfv-ssl-aux,direct,"0.850 [0.815, 0.883]","0.885 [0.861, 0.908]","0.881 [0.855, 0.905]",0.767,0.867,0.917,785,0.868,0.803,0.793,0.853,0.861,0.807,0.333,diffuse | val visit 0.877
3,bfv-kd,direct,"0.854 [0.820, 0.886]","0.885 [0.861, 0.908]","0.881 [0.855, 0.905]",0.768,0.873,0.922,785,0.870,0.809,0.791,0.858,0.861,0.825,0.333,diffuse | val visit 0.877
4,ENSEMBLE direct,direct,"0.854 [0.819, 0.885]","0.882 [0.857, 0.907]","0.881 [0.854, 0.904]",0.770,0.874,0.916,785,0.869,0.812,0.787,0.857,0.859,0.827,0.500,bfv-pair+bfv-base+bfv-ssl-aux+bfv-kd
5,tier1-pair,tier 1 alone (viral vs rest),"0.901 [0.859, 0.940]","0.929 [0.901, 0.954]","0.930 [0.901, 0.957]",NaN,NaN,NaN,785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary AUROC
6,tier1-pair -> bf-pair,two-tier,"0.825 [0.792, 0.854]","0.860 [0.834, 0.887]","0.858 [0.831, 0.884]",0.728,0.845,0.901,785,0.837,0.791,0.791,0.827,0.829,0.794,0.667,
7,tier1-pair -> bf-pair-aux,two-tier,"0.832 [0.800, 0.860]","0.862 [0.836, 0.889]","0.863 [0.838, 0.890]",0.742,0.852,0.901,785,0.849,0.783,0.809,0.833,0.836,0.811,0.833,
8,tier1-pair -> bf-ssl-pair-aux,two-tier,"0.837 [0.805, 0.868]","0.876 [0.849, 0.901]","0.870 [0.842, 0.896]",0.746,0.864,0.901,785,0.851,0.795,0.804,0.839,0.843,0.814,0.667,
9,v6 bfv-clin rescored on v7 rows,direct,"0.825 [0.791, 0.855]","0.852 [0.824, 0.882]","0.851 [0.818, 0.880]",0.741,0.852,0.882,785,0.844,0.772,0.737,0.830,0.830,0.804,0.500,"v6 training, same rows"


### 5c. Cross-validation: pooled out-of-fold predictions and paired differences

In [13]:
# ===== 5c Cross-validation =====
cv_runs = [t for t in DONE if t.startswith('cv-')]
cv_table, cv_oof = pd.DataFrame(), {}
if cv_runs:
    recipes = sorted({t.rsplit('-f', 1)[0] for t in cv_runs})
    rows = []
    for rcp in recipes:
        folds = sorted(t for t in cv_runs if t.startswith(rcp + '-f'))
        oof = pd.concat([preds(t) for t in folds], ignore_index=True)
        assert not oof['image_name'].duplicated().any(), f'{rcp}: overlapping folds'
        cv_oof[rcp] = oof
        r = score(oof, CLASSES['bf'], YMAP['bf'])
        rows.append(dict(recipe=rcp, source=CV_RECIPES.get(rcp, '') if 'CV_RECIPES' in globals() else '', folds=len(folds),
                         index_visit=fmt(r['index_visit']), patient=fmt(r['patient']), visit=fmt(r['visit']),
                         n_visit=r['visit']['n'], n_pat=r['patient']['n']))
    cv_table = pd.DataFrame(rows); cv_table.to_csv(f'{RUNS_DIR}/results_cv.csv', index=False)
    print('Five-fold CV over all patients (pooled out-of-fold AUROC):'); display(cv_table)
    if 'cv-base' in cv_oof:
        base_v = levels(cv_oof['cv-base'].pipe(lambda d: d[d.label.isin(YMAP['bf'])]), CLASSES['bf'])
        for rcp in recipes:
            if rcp == 'cv-base': continue
            diff_rows = []
            for lvl in ('index_visit', 'patient', 'visit'):
                a = base_v[lvl].set_index(VKEYS if lvl != 'patient' else ['patient_id'])
                b = levels(cv_oof[rcp].pipe(lambda d: d[d.label.isin(YMAP['bf'])]), CLASSES['bf'])[lvl].set_index(VKEYS if lvl != 'patient' else ['patient_id']).reindex(a.index)
                y = a['label'].map(YMAP['bf']).values
                grp = a.index.get_level_values('patient_id') if lvl != 'patient' else a.index.values
                dlt, ci, win = paired_diff(y, P_of(a, CLASSES['bf']), P_of(b, CLASSES['bf']), grp)
                diff_rows.append(dict(level=lvl, delta=round(dlt, 4), ci_lo=round(ci[0], 4), ci_hi=round(ci[1], 4), p_better=round(win, 3)))
            print(f'\n{rcp} minus cv-base (paired patient bootstrap):'); display(pd.DataFrame(diff_rows))
else:
    print('no CV runs in the ledger')

Five-fold CV over all patients (pooled out-of-fold AUROC):


,recipe,source,folds,index_visit,patient,visit,n_visit,n_pat
0,cv-base,,5,"0.837 [0.821, 0.852]","0.838 [0.822, 0.852]","0.813 [0.796, 0.828]",4693,2620
1,cv-best,,5,"0.887 [0.873, 0.899]","0.884 [0.871, 0.897]","0.867 [0.853, 0.881]",4693,2620
2,cv-pair,,5,"0.859 [0.843, 0.873]","0.856 [0.841, 0.871]","0.837 [0.822, 0.853]",4693,2620



cv-best minus cv-base (paired patient bootstrap):


,level,delta,ci_lo,ci_hi,p_better
0,index_visit,0.0505,0.0397,0.0617,1.0
1,patient,0.0460,0.0356,0.0564,1.0
2,visit,0.0542,0.0436,0.0646,1.0



cv-pair minus cv-base (paired patient bootstrap):


,level,delta,ci_lo,ci_hi,p_better
0,index_visit,0.0224,0.0108,0.0339,1.0
1,patient,0.0176,0.0066,0.0290,1.0
2,visit,0.0246,0.0130,0.0362,1.0


In [7]:
oof = cv_oof['cv-best'] if 'cv-best' in cv_oof else cv_oof['cv-base']
v = levels(oof[oof.label.isin(YMAP['bf'])], CLASSES['bf'])['index_visit'].copy()
v['y'] = v['label'].map(YMAP['bf']); v['conf'] = (v['prob_Fungal'] - 0.5).abs(); v = v.sort_values('conf', ascending=False)
for cov in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
    s = v.head(int(cov * len(v))); lo, hi = boot_ci(s.y, P_of(s, CLASSES['bf']), s.patient_id.values, n_boot=300)
    print(f'coverage {cov:.0%}: n={len(s):4d}  index-visit AUROC {auroc(s.y, P_of(s, CLASSES["bf"])):.3f} [{lo:.3f}, {hi:.3f}]  acc {((s.prob_Fungal >= 0.5) == s.y).mean():.3f}')

coverage 40%: n=1049  index-visit AUROC 0.957 [0.943, 0.969]  acc 0.942
coverage 50%: n=1311  index-visit AUROC 0.946 [0.934, 0.959]  acc 0.922
coverage 60%: n=1573  index-visit AUROC 0.935 [0.922, 0.947]  acc 0.908
coverage 70%: n=1836  index-visit AUROC 0.925 [0.910, 0.938]  acc 0.891
coverage 80%: n=2098  index-visit AUROC 0.914 [0.901, 0.924]  acc 0.872
coverage 90%: n=2360  index-visit AUROC 0.901 [0.890, 0.914]  acc 0.846
coverage 100%: n=2623  index-visit AUROC 0.887 [0.873, 0.899]  acc 0.814
